### Imports 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
import ipywidgets as widgets
import joblib
from IPython.display import display
from neo.io import AxonIO 
from neo.io import Spike2IO
import sys 
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
from os import chdir
import pandas as pd
from scipy.signal import butter, sosfiltfilt, find_peaks
from pathlib import Path
from scipy.signal import find_peaks
from scipy.signal import butter, sosfiltfilt
from scipy.ndimage import binary_dilation
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm
import pyqtgraph as pg
from pathlib import Path
from PyQt6 import QtCore, QtWidgets, QtGui
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    train_test_split,
)
from scipy.ndimage import (
    gaussian_filter1d,
    binary_closing,
    binary_opening,
)
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
)
from sklearn.model_selection import (
    GroupKFold,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


chdir('/Volumes/marder-lab/helmore/marder_py')

%matplotlib qt
%gui qt

### Read me 

- Ensure your directory is properly set before running any code, All outputs will be saved there 
- Steps 1-3 can be run without editing 
- Step 5 noise filter requires fine tuning settings for proper threshold 
- Steps 6 and 7 have sections that require user settings editing noted in sub headers 
- Step 8 requires denoting file path and sequence for plotting final viewer gui 


### Step 0: Set file path 

In [ ]:
file_path = Path('/Volumes/marder-lab/Maine Data/Acclimation Data Summer 2026/Rig 6/07_01_26/MostModulatorsCutShort.smr')

### Step 1: Load recording file here 

In [ ]:

reader = Spike2IO(filename=str(file_path))
block = reader.read_block(lazy=False)

for segment_index, segment in enumerate(block.segments):
    print(f"\nSegment {segment_index}")
    print(f"Marker/event channels: {len(segment.events)}")

    for event_index, event in enumerate(segment.events):
        print(
            f"  {event_index}:",
            f"name={event.name!r},",
            f"markers={len(event)},",
            f"annotations={event.annotations}",
        )

print(f"Segments found: {len(block.segments)}")

# Use the first recording segment
segment = block.segments[0]

print(f"Analog signal groups: {len(segment.analogsignals)}")
print(f"Event groups: {len(segment.events)}")
print(f"Spike trains: {len(segment.spiketrains)}")

### Step 2: Extract channels

In [ ]:
channels = []

for signal_group_index, signal_group in enumerate(
    segment.analogsignals
):
    data = np.asarray(signal_group, dtype=float)

    # Neo normally represents data as:
    # rows = samples, columns = channels
    if data.ndim == 1:
        data = data[:, np.newaxis]

    times = np.asarray(
        signal_group.times.rescale("s").magnitude,
        dtype=float,
    )

    channel_names = signal_group.array_annotations.get(
        "channel_names"
    )

    for column in range(data.shape[1]):
        if (
            channel_names is not None
            and column < len(channel_names)
        ):
            name = str(channel_names[column])

        elif signal_group.name:
            if data.shape[1] == 1:
                name = str(signal_group.name)
            else:
                name = f"{signal_group.name} {column + 1}"

        else:
            name = (
                f"Signal {signal_group_index + 1}, "
                f"channel {column + 1}"
            )

        channels.append({
            "name": name,
            "time": times,
            "values": data[:, column],
            "units": str(signal_group.units),
        })

if not channels:
    raise ValueError(
        "No continuous analog channels were found in the file."
    )

print(f"Extracted {len(channels)} channels")

for index, channel in enumerate(channels):
    print(
        index,
        channel["name"],
        f"{len(channel['values'])} samples",
        channel["units"],
    )

### Optional: Extract text markers

In [ ]:
for segment_index, segment in enumerate(block.segments):
    for event in segment.events:
        times = np.asarray(
            event.times.rescale("s").magnitude
        )

        labels = np.asarray(event.labels)

        print(f"\nMarker channel: {event.name}")

        for time, label in zip(times, labels):
            # Decode labels stored as bytes
            if isinstance(label, bytes):
                label = label.decode(
                    "utf-8",
                    errors="replace",
                )

            print(f"{time:.4f} s: {label}")

### Step 3: Preview recording file 

#### create viewer

In [ ]:
# Number of seconds shown at one time
window_seconds = 15.0

recording_start = min(
    channel["time"][0] for channel in channels
)

recording_end = max(
    channel["time"][-1] for channel in channels
)

recording_duration = recording_end - recording_start

if recording_duration <= 0:
    raise ValueError("The recording has no measurable duration.")

# Slider operates in milliseconds
start_ms = int(recording_start * 1000)

latest_window_start = max(
    recording_start,
    recording_end - window_seconds,
)

end_ms = int(latest_window_start * 1000)



time_slider = widgets.IntSlider(
    value=start_ms,
    min=start_ms,
    max=max(start_ms, end_ms),
    step=100,
    description="Time:",
    continuous_update=True,
    readout=False,
    layout=widgets.Layout(width="700px"),
)

time_label = widgets.Label()

play_button = widgets.Play(
    value=start_ms,
    min=start_ms,
    max=max(start_ms, end_ms),
    step=100,
    interval=100,
    description="Play",
)

widgets.jslink(
    (play_button, "value"),
    (time_slider, "value"),
)

y_zoom_sliders = []

for channel in channels:
    slider = widgets.FloatLogSlider(
        value=1.0,
        base=10,
        min=-1,       # 0.1×: zoom out
        max=2,        # 100×: zoom in
        step=0.05,
        description=channel["name"][:18],
        readout_format=".2f",
        continuous_update=True,
        style={"description_width": "150px"},
        layout=widgets.Layout(width="450px"),
    )

    y_zoom_sliders.append(slider)


# Create plots


colors = plt.cm.tab20(
    np.linspace(0, 1, max(20, len(channels)))
)

plt.ioff()

fig, axes = plt.subplots(
    nrows=len(channels),
    ncols=1,
    figsize=(12, max(2.2 * len(channels), 4)),
    sharex=True,
    squeeze=False,
)

axes = axes[:, 0]
lines = []

for index, (axis, channel) in enumerate(
    zip(axes, channels)
):
    color = colors[index % len(colors)]

    line, = axis.plot(
        [],
        [],
        color=color,
        linewidth=0.8,
    )

    axis.set_ylabel(
        f"{channel['name']}\n{channel['units']}",
        color=color,
    )

    axis.tick_params(
        axis="y",
        labelcolor=color,
    )

    axis.grid(
        visible=True,
        alpha=0.25,
    )

    lines.append(line)

axes[-1].set_xlabel("Time (seconds)")

fig.suptitle(file_path.name)
fig.tight_layout()


def update_view(change=None):
    start_time = time_slider.value / 1000
    stop_time = min(
        start_time + window_seconds,
        recording_end,
    )

    time_label.value = (
        f"Showing {start_time:.2f}–{stop_time:.2f} seconds"
    )

    for index, (axis, line, channel) in enumerate(
        zip(axes, lines, channels)
    ):
        time = channel["time"]
        values = channel["values"]

        mask = (
            (time >= start_time)
            & (time <= stop_time)
        )

        visible_time = time[mask]
        visible_values = values[mask]

        # Downsample only for screen rendering
        max_display_points = 10_000

        if len(visible_time) > max_display_points:
            stride = int(
                np.ceil(
                    len(visible_time)
                    / max_display_points
                )
            )

            display_time = visible_time[::stride]
            display_values = visible_values[::stride]

        else:
            display_time = visible_time
            display_values = visible_values

        line.set_data(
            display_time,
            display_values,
        )

        axis.set_xlim(
            start_time,
            stop_time,
        )

     
        finite_values = visible_values[
            np.isfinite(visible_values)
        ]

        if len(finite_values):
            low, high = np.percentile(
                finite_values,
                [1, 99],
            )

            center = (low + high) / 2
            half_range = (high - low) / 2

            if half_range <= 0:
                half_range = max(
                    abs(center) * 0.05,
                    1.0,
                )

            # Larger values zoom in;
            # smaller values zoom out
            zoom = y_zoom_sliders[index].value
            displayed_half_range = half_range / zoom

            axis.set_ylim(
                center - displayed_half_range,
                center + displayed_half_range,
            )

    fig.canvas.draw_idle()




time_slider.observe(
    update_view,
    names="value",
)

for slider in y_zoom_sliders:
    slider.observe(
        update_view,
        names="value",
    )


update_view()


time_controls = widgets.VBox([
    widgets.HBox([
        play_button,
        time_slider,
    ]),
    time_label,
])

y_controls = widgets.VBox([
    widgets.HTML(
        "<h4>Y-axis zoom by channel</h4>"
        "<p>Above 1 zooms in; below 1 zooms out.</p>"
    ),
    *y_zoom_sliders,
])

display(time_controls)
display(y_controls)
#display(fig.canvas) use ths line instead for an inline viewer 
plt.show(block=False)

plt.ion()

### Step 5: Filter Noise 

In [ ]:


# User setting section, edit channels to analyze and artifact thresholds 


# Select channels by name or index.
# Examples:
# channels_to_analyze = ["Orange", "Green", "Black"]
# channels_to_analyze = [2, 3, 6]
channels_to_analyze = [
    "Red",
    "Lavender",
    "Yellow",
    "Orange",
    "Green",
    "Blue",
    "Purple",
    "Black",
]

# Artifact cutoff for each channel in millivolts.
#
# The cutoff is applied to the ABSOLUTE deviation from the
# channel median. Thus 28 mV removes excursions above +28 mV
# or below -28 mV relative to the median.
#
# These keys should match your channel names.
artifact_thresholds_mV = {
    "Red": 120.0,
    "Lavender": 50.0,
    "Yellow": 20.0,
    "Orange": 30.0,
    "Green": 1400.0,
    "Blue": 45.0,
    "Purple": 28.0,
    "Black": 35.0,
}


default_artifact_threshold_mV = 28.0

# Adding this much time before and after every artifact.
# helps remove filter ringing around extreme events.
artifact_padding_ms = 10.0

#bandpass filt
filter_low_hz = 100.0
filter_high_hz = 1500.0
filter_order = 3


preview_start_s = None
preview_duration_s = 120.0
final_plot_start_s = None
final_plot_duration_s = 30.0


show_raw_preview = True
show_final_plot = True


def find_channel(channel_identifier, all_channels):
    """
    Find a channel by integer index or channel name.
    """

    if isinstance(channel_identifier, int):
        if not 0 <= channel_identifier < len(all_channels):
            raise IndexError(
                f"Channel index {channel_identifier} "
                f"is outside 0–{len(all_channels) - 1}."
            )

        return channel_identifier, all_channels[channel_identifier]

    requested_name = str(channel_identifier).strip().lower()

    # Try an exact name match first
    exact_matches = [
        (index, channel)
        for index, channel in enumerate(all_channels)
        if str(channel["name"]).strip().lower()
        == requested_name
    ]

    if len(exact_matches) == 1:
        return exact_matches[0]

    partial_matches = [
        (index, channel)
        for index, channel in enumerate(all_channels)
        if requested_name
        in str(channel["name"]).strip().lower()
    ]

    if len(partial_matches) == 1:
        return partial_matches[0]

    if len(partial_matches) > 1:
        matching_names = [
            channel["name"]
            for _, channel in partial_matches
        ]

        raise ValueError(
            f"{channel_identifier!r} matched multiple channels: "
            f"{matching_names}"
        )

    available_names = [
        channel["name"] for channel in all_channels
    ]

    raise ValueError(
        f"Could not find channel {channel_identifier!r}. "
        f"Available channels: {available_names}"
    )


def threshold_mV_to_native_units(
    threshold_mV,
    units,
):
    """
    Convert an artifact threshold in mV to the native units
    used in the channel array.
    """

    units_text = (
        str(units)
        .lower()
        .replace("µ", "u")
        .replace("μ", "u")
        .replace(" ", "")
    )

    if "mv" in units_text:
        # Data values are already millivolts
        return threshold_mV

    if "uv" in units_text:
        # 1 mV = 1000 uV
        return threshold_mV * 1000.0

    if "v" in units_text:
        # 1 mV = 0.001 V
        return threshold_mV / 1000.0

    raise ValueError(
        f"Could not interpret channel units {units!r}. "
        "Expected V, mV, or uV."
    )


def robust_noise_estimate(signal):
    """
    Estimate background scale using median absolute deviation.
    """

    signal = np.asarray(signal, dtype=float)

    center = np.nanmedian(signal)

    mad_sigma = (
        np.nanmedian(
            np.abs(signal - center)
        )
        / 0.6745
    )

    if not np.isfinite(mad_sigma) or mad_sigma <= 0:
        raise ValueError(
            "Could not calculate a positive robust noise "
            "estimate for this channel."
        )

    return center, mad_sigma


def make_artifact_mask(
    time,
    raw_signal,
    threshold_native,
    padding_ms,
):
    """
    Identify samples whose absolute deviation from the median
    exceeds the physical amplitude threshold.
    """

    time = np.asarray(time, dtype=float)
    raw_signal = np.asarray(raw_signal, dtype=float)

    sampling_rate = 1 / np.median(
        np.diff(time)
    )

    center, noise_sigma = robust_noise_estimate(
        raw_signal
    )

    centered_signal = raw_signal - center

    # Convert the physical cutoff to robust-Z units
    artifact_z_threshold = (
        threshold_native / noise_sigma
    )

    robust_amplitude = (
        np.abs(centered_signal)
        / noise_sigma
    )

    initial_mask = (
        np.abs(centered_signal)
        >= threshold_native
    )

    padding_samples = max(
        1,
        int(
            sampling_rate
            * padding_ms
            / 1000
        ),
    )

    padded_mask = binary_dilation(
        initial_mask,
        iterations=padding_samples,
    )

    return {
        "center": center,
        "noise_sigma": noise_sigma,
        "artifact_z_threshold": artifact_z_threshold,
        "robust_amplitude": robust_amplitude,
        "initial_mask": initial_mask,
        "artifact_mask": padded_mask,
        "sampling_rate": sampling_rate,
        "padding_samples": padding_samples,
    }


def interpolate_masked_samples(
    signal,
    mask,
):
    """
    Linearly interpolate across artifact intervals.

    This interpolated signal is used only as input to the
    filter. The final masked filtered signal still contains
    NaN gaps at excluded locations.
    """

    signal = np.asarray(signal, dtype=float)
    mask = np.asarray(mask, dtype=bool)

    sample_indices = np.arange(len(signal))

    good_samples = (
        ~mask
        & np.isfinite(signal)
    )

    if np.sum(good_samples) < 2:
        raise ValueError(
            "Too few unmasked samples remain for interpolation."
        )

    interpolated = signal.copy()

    interpolated[mask] = np.interp(
        sample_indices[mask],
        sample_indices[good_samples],
        signal[good_samples],
    )

    return interpolated


def count_masked_events(mask):
    """
    Count contiguous masked intervals.
    """

    changes = np.diff(
        np.concatenate([
            [False],
            np.asarray(mask, dtype=bool),
            [False],
        ]).astype(int)
    )

    starts = np.flatnonzero(changes == 1)
    stops = np.flatnonzero(changes == -1)

    return len(starts), starts, stops


selected_channels = []

for requested_channel in channels_to_analyze:
    channel_index, channel = find_channel(
        requested_channel,
        channels,
    )

    selected_channels.append({
        "index": channel_index,
        "channel": channel,
    })

print("Selected channels:")

for item in selected_channels:
    print(
        f"  index={item['index']}, "
        f"name={item['channel']['name']!r}, "
        f"units={item['channel']['units']}"
    )



processed_channels = {}

for item in selected_channels:
    channel_index = item["index"]
    channel = item["channel"]
    channel_name = str(channel["name"])

    time = np.asarray(
        channel["time"],
        dtype=float,
    )

    raw_signal = np.asarray(
        channel["values"],
        dtype=float,
    )

  
    threshold_mV = artifact_thresholds_mV.get(
        channel_name,
        default_artifact_threshold_mV,
    )

    threshold_native = threshold_mV_to_native_units(
        threshold_mV=threshold_mV,
        units=channel["units"],
    )

    mask_result = make_artifact_mask(
        time=time,
        raw_signal=raw_signal,
        threshold_native=threshold_native,
        padding_ms=artifact_padding_ms,
    )

    artifact_mask = mask_result["artifact_mask"]
    sampling_rate = mask_result["sampling_rate"]

   
    centered_signal = (
        raw_signal
        - mask_result["center"]
    )

    interpolation_signal = (
        interpolate_masked_samples(
            signal=centered_signal,
            mask=artifact_mask,
        )
    )


    actual_high_hz = min(
        filter_high_hz,
        sampling_rate * 0.45,
    )

    if filter_low_hz >= actual_high_hz:
        raise ValueError(
            f"Channel {channel_name!r} has a sampling rate "
            f"of {sampling_rate:.2f} Hz. The requested filter "
            f"{filter_low_hz}–{actual_high_hz:.2f} Hz is invalid. "
            "Reduce filter_low_hz."
        )

    sos = butter(
        N=filter_order,
        Wn=[
            filter_low_hz,
            actual_high_hz,
        ],
        btype="bandpass",
        fs=sampling_rate,
        output="sos",
    )

    filtered_interpolated = sosfiltfilt(
        sos,
        interpolation_signal,
    )

  
    filtered_masked = (
        filtered_interpolated.copy()
    )

    filtered_masked[artifact_mask] = np.nan

    event_count, event_starts, event_stops = (
        count_masked_events(artifact_mask)
    )

    processed_channels[channel_name] = {
        "channel_index": channel_index,
        "units": channel["units"],
        "time": time,
        "raw_signal": raw_signal,
        "centered_signal": centered_signal,
        "interpolated_signal": interpolation_signal,
        "filtered_interpolated": filtered_interpolated,
        "filtered_signal": filtered_masked,
        "artifact_mask": artifact_mask,
        "initial_artifact_mask": mask_result["initial_mask"],
        "robust_amplitude": mask_result["robust_amplitude"],
        "robust_noise": mask_result["noise_sigma"],
        "artifact_threshold_mV": threshold_mV,
        "artifact_threshold_native": threshold_native,
        "artifact_z_threshold": (
            mask_result["artifact_z_threshold"]
        ),
        "sampling_rate": sampling_rate,
        "filter_low_hz": filter_low_hz,
        "filter_high_hz": actual_high_hz,
        "artifact_event_count": event_count,
        "artifact_event_starts": event_starts,
        "artifact_event_stops": event_stops,
    }

    print()
    print(f"Channel: {channel_name}")
    print(
        f"  Sampling rate: "
        f"{sampling_rate:.2f} Hz"
    )
    print(
        f"  Robust noise: "
        f"{mask_result['noise_sigma']:.8f} "
        f"{channel['units']}"
    )
    print(
        f"  Physical artifact cutoff: "
        f"{threshold_mV:.3f} mV"
    )
    print(
        f"  Equivalent robust-Z cutoff: "
        f"{mask_result['artifact_z_threshold']:.2f}"
    )
    print(
        f"  Artifact intervals: "
        f"{event_count}"
    )
    print(
        f"  Recording excluded: "
        f"{artifact_mask.mean() * 100:.5f}%"
    )
    print(
        f"  Filter: "
        f"{filter_low_hz:.1f}–"
        f"{actual_high_hz:.1f} Hz"
    )


if show_raw_preview:
    figure, axes = plt.subplots(
        len(selected_channels),
        1,
        figsize=(
            16,
            max(3 * len(selected_channels), 5),
        ),
        sharex=True,
        squeeze=False,
    )

    axes = axes[:, 0]

    for axis, item in zip(
        axes,
        selected_channels,
    ):
        channel_name = str(
            item["channel"]["name"]
        )

        result = processed_channels[channel_name]

        time = result["time"]
        raw_signal = result["raw_signal"]

        if preview_start_s is None:
            plot_start = time[0]
        else:
            plot_start = preview_start_s

        plot_stop = min(
            plot_start + preview_duration_s,
            time[-1],
        )

        plot_mask = (
            (time >= plot_start)
            & (time <= plot_stop)
        )

        center = np.nanmedian(raw_signal)
        threshold_native = result[
            "artifact_threshold_native"
        ]

        axis.plot(
            time[plot_mask],
            raw_signal[plot_mask],
            color="black",
            linewidth=0.5,
        )

        axis.axhline(
            center + threshold_native,
            color="red",
            linestyle="--",
            linewidth=1,
            label=(
                f"±{result['artifact_threshold_mV']:.1f} mV"
            ),
        )

        axis.axhline(
            center - threshold_native,
            color="red",
            linestyle="--",
            linewidth=1,
        )

        axis.set_ylabel(
            f"{channel_name}\n{result['units']}"
        )

        axis.grid(alpha=0.2)
        axis.legend(loc="upper right")

    axes[-1].set_xlabel("Time (seconds)")

    figure.suptitle(
        "Long raw preview with proposed artifact cutoffs"
    )

    figure.tight_layout()
    plt.show()


if show_final_plot:
    figure, axes = plt.subplots(
        len(selected_channels),
        2,
        figsize=(
            18,
            max(3.5 * len(selected_channels), 6),
        ),
        sharex="col",
        squeeze=False,
    )

    for row, item in enumerate(selected_channels):
        channel_name = str(
            item["channel"]["name"]
        )

        result = processed_channels[channel_name]

        time = result["time"]
        raw_signal = result["raw_signal"]
        filtered_signal = result["filtered_signal"]
        artifact_mask = result["artifact_mask"]

        if final_plot_start_s is None:
            plot_start = time[0]
        else:
            plot_start = final_plot_start_s

        plot_stop = min(
            plot_start + final_plot_duration_s,
            time[-1],
        )

        plot_mask = (
            (time >= plot_start)
            & (time <= plot_stop)
        )

        excluded_in_window = (
            plot_mask
            & artifact_mask
        )

        # --------------------------------
        # Left: original raw signal
        # --------------------------------

        raw_axis = axes[row, 0]

        raw_axis.plot(
            time[plot_mask],
            raw_signal[plot_mask],
            color="black",
            linewidth=0.6,
            label="Raw signal",
        )

        raw_axis.scatter(
            time[excluded_in_window],
            raw_signal[excluded_in_window],
            color="red",
            s=10,
            zorder=3,
            label="Excluded",
        )

        raw_axis.set_ylabel(
            f"{channel_name}\n{result['units']}"
        )

        raw_axis.set_title(
            f"Raw: {channel_name}"
        )

        raw_axis.grid(alpha=0.2)
        raw_axis.legend(loc="upper right")

        # --------------------------------
        # Right: cleaned filtered signal
        # --------------------------------

        filtered_axis = axes[row, 1]

        filtered_axis.plot(
            time[plot_mask],
            filtered_signal[plot_mask],
            color="blue",
            linewidth=0.6,
            label="Filtered retained signal",
        )

        # Shade excluded intervals
        mask_in_plot = artifact_mask[plot_mask]
        time_in_plot = time[plot_mask]

        if np.any(mask_in_plot):
            filtered_axis.fill_between(
                time_in_plot,
                0,
                1,
                where=mask_in_plot,
                transform=(
                    filtered_axis
                    .get_xaxis_transform()
                ),
                color="red",
                alpha=0.2,
                label="Excluded interval",
            )

        filtered_axis.set_title(
            f"Filtered: {result['filter_low_hz']:.0f}–"
            f"{result['filter_high_hz']:.0f} Hz"
        )

        filtered_axis.grid(alpha=0.2)
        filtered_axis.legend(loc="upper right")

    axes[-1, 0].set_xlabel("Time (seconds)")
    axes[-1, 1].set_xlabel("Time (seconds)")

    figure.suptitle(
        f"{final_plot_duration_s:.0f}-second artifact review"
    )

    figure.tight_layout()
    plt.show()


# working ex

print()
print("Processing complete.")
print(
    "Available results:",
    list(processed_channels.keys()),
)

print(
    "\nExample access:\n"
    "result = processed_channels['Orange']\n"
    "time = result['time']\n"
    "filtered_signal = result['filtered_signal']\n"
    "artifact_mask = result['artifact_mask']"
)

### Step 6: Create Spike Classifer [Follow "A" for first file and "B" for sebsequent file(s)]

yellow - son\
black - ion\
blue - mvn\
green - pdn\
red - pyn


In [ ]:
channel_name = "Red"
# Reference channel shown in the lower panel.
# This channel is for visual/rhythmic context only.
reference_channel_name = "Green"
exp = "SC0005"
signal = "PY"
# Labels and desired number of examples
label_targets = {
    "PY": 25,
    "LP":25,
    "noise": 25,
}               

# Seconds displayed around each candidate spike
context_before_seconds = 5.0
context_after_seconds = 5.0

#### 6.1: generating samples for manual labelling

In [ ]:
# Spike detection driver (note this can be used on its own outside the more advanced spike detect pipeline)

def estimate_background_noise(
    values,
    quantile=0.25,
):
    """
    Estimate the center and background-noise scale of a signal.

    Using the lower portion of the amplitude distribution makes
    the estimate less sensitive to genuine spikes and bursts.
    """

    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) < 10:
        return np.nan, np.nan

    if not 0 < quantile < 1:
        raise ValueError(
            "quantile must be between 0 and 1."
        )

    center = np.median(values)

    absolute_deviation = np.abs(
        values - center
    )

    # For a standard normal distribution:
    # P(|Z| <= x) = quantile
    scale_constant = norm.ppf(
        (1 + quantile) / 2
    )

    noise_sigma = (
        np.quantile(
            absolute_deviation,
            quantile,
        )
        / scale_constant
    )

    # Fall back to conventional MAD if necessary
    if (
        not np.isfinite(noise_sigma)
        or noise_sigma <= 0
    ):
        noise_sigma = (
            np.median(
                absolute_deviation
            )
            / 0.6745
        )

    return center, noise_sigma


# Adaptive spike detector 


def detect_spikes_processed_channel_adaptive(
    result,
    threshold_factor=2,
    upper_threshold_factor=None,
    shadow_period_ms=1.0,
    prominence_factor=None,
    polarity="both",
    adaptive_window_s=30.0,
    adaptive_step_s=15.0,
    noise_quantile=0.25,
    minimum_valid_fraction=0.25,
    noise_scale_bounds=(0.25, 4.0),
):
    """
    Detect extracellular spikes using locally adaptive noise,
    threshold, and prominence estimates.

    Parameters
    ----------
    result : dict
        One channel from processed_channels.

    threshold_factor : float
        Lower detection threshold in local noise units.

    upper_threshold_factor : float or None
        Optional upper threshold in local noise units.

    shadow_period_ms : float
        Minimum time between detected events.

    prominence_factor : float or None
        Optional minimum prominence in local noise units.

    polarity : str
        Which spike direction to detect:

            "positive"
            "negative"
            "both"

    adaptive_window_s : float
        Duration of each local noise-estimation window.

    adaptive_step_s : float
        Time between consecutive window centers.

        For example, a 30-second window with a 15-second step
        gives 50% overlap.

    noise_quantile : float
        Portion of the absolute-amplitude distribution used to
        estimate background noise.

    minimum_valid_fraction : float
        Minimum valid fraction required in each adaptive window.

    noise_scale_bounds : tuple or None
        Lower and upper bounds for local noise relative to the
        global noise estimate.

        For example, (0.25, 4.0) prevents the local threshold
        from falling below one-quarter or rising above four
        times the global estimate.

    Returns
    -------
    dict
        Detected spikes, adaptive thresholds, noise estimates,
        diagnostic information, and a spike DataFrame.
    """



    polarity = str(
        polarity
    ).strip().lower()

    allowed_polarities = {
        "positive",
        "negative",
        "both",
    }

    if polarity not in allowed_polarities:
        raise ValueError(
            "polarity must be 'positive', "
            "'negative', or 'both'."
        )

    if threshold_factor <= 0:
        raise ValueError(
            "threshold_factor must be positive."
        )

    if upper_threshold_factor is not None:
        if (
            upper_threshold_factor
            <= threshold_factor
        ):
            raise ValueError(
                "upper_threshold_factor must be greater "
                "than threshold_factor."
            )

    if shadow_period_ms <= 0:
        raise ValueError(
            "shadow_period_ms must be positive."
        )

    if adaptive_window_s <= 0:
        raise ValueError(
            "adaptive_window_s must be positive."
        )

    if adaptive_step_s <= 0:
        raise ValueError(
            "adaptive_step_s must be positive."
        )



    time = np.asarray(
        result["time"],
        dtype=float,
    )

    filtered_signal = np.asarray(
        result["filtered_signal"],
        dtype=float,
    )

    artifact_mask = np.asarray(
        result["artifact_mask"],
        dtype=bool,
    )

    sampling_rate = float(
        result["sampling_rate"]
    )

    if not (
        len(time)
        == len(filtered_signal)
        == len(artifact_mask)
    ):
        raise ValueError(
            "Time, filtered signal, and artifact mask "
            "must have equal lengths."
        )

    if len(time) < 10:
        raise ValueError(
            "The recording contains too few samples."
        )

    if np.any(np.diff(time) <= 0):
        raise ValueError(
            "The time array must be strictly increasing."
        )

    valid_samples = (
        np.isfinite(filtered_signal)
        & ~artifact_mask
    )

    clean_values = filtered_signal[
        valid_samples
    ]

    if len(clean_values) < 10:
        raise ValueError(
            "Too few valid samples remain for detection."
        )

   
    # Calculating global noise estimate
    

    (
        global_center,
        global_noise_sigma,
    ) = estimate_background_noise(
        clean_values,
        quantile=noise_quantile,
    )

    if (
        not np.isfinite(global_noise_sigma)
        or global_noise_sigma <= 0
    ):
        raise ValueError(
            "Could not estimate the global noise level."
        )


    # Calculating local noise in overlapping windows


    recording_start = time[0]
    recording_stop = time[-1]

    window_starts = np.arange(
        recording_start,
        recording_stop,
        adaptive_step_s,
    )

    if len(window_starts) == 0:
        window_starts = np.array([
            recording_start
        ])

    window_centers = []
    local_centers = []
    local_noise_values = []
    valid_counts = []

    expected_samples_per_window = max(
        1,
        int(
            adaptive_window_s
            * sampling_rate
        ),
    )

    minimum_valid_samples = max(
        10,
        int(
            expected_samples_per_window
            * minimum_valid_fraction
        ),
    )

    for window_start in window_starts:
        window_stop = min(
            window_start + adaptive_window_s,
            recording_stop,
        )

        left_index = np.searchsorted(
            time,
            window_start,
            side="left",
        )

        right_index = np.searchsorted(
            time,
            window_stop,
            side="right",
        )

        window_valid = valid_samples[
            left_index:right_index
        ]

        window_values = filtered_signal[
            left_index:right_index
        ][window_valid]

        window_center_time = (
            window_start + window_stop
        ) / 2

        window_centers.append(
            window_center_time
        )

        valid_counts.append(
            len(window_values)
        )

        if (
            len(window_values)
            < minimum_valid_samples
        ):
            local_centers.append(
                np.nan
            )

            local_noise_values.append(
                np.nan
            )

            continue

        (
            local_center,
            local_sigma,
        ) = estimate_background_noise(
            window_values,
            quantile=noise_quantile,
        )

        local_centers.append(
            local_center
        )

        local_noise_values.append(
            local_sigma
        )

    window_centers = np.asarray(
        window_centers,
        dtype=float,
    )

    local_centers = np.asarray(
        local_centers,
        dtype=float,
    )

    local_noise_values = np.asarray(
        local_noise_values,
        dtype=float,
    )

    valid_counts = np.asarray(
        valid_counts,
        dtype=int,
    )


    valid_noise_windows = (
        np.isfinite(local_noise_values)
        & (local_noise_values > 0)
    )

    if not np.any(valid_noise_windows):
        local_noise_values[:] = (
            global_noise_sigma
        )

    else:
        local_noise_values = np.interp(
            window_centers,
            window_centers[
                valid_noise_windows
            ],
            local_noise_values[
                valid_noise_windows
            ],
        )

    valid_center_windows = np.isfinite(
        local_centers
    )

    if not np.any(valid_center_windows):
        local_centers[:] = (
            global_center
        )

    else:
        local_centers = np.interp(
            window_centers,
            window_centers[
                valid_center_windows
            ],
            local_centers[
                valid_center_windows
            ],
        )

    raw_local_noise_values = (
        local_noise_values.copy()
    )

 
    if noise_scale_bounds is not None:
        (
            minimum_noise_scale,
            maximum_noise_scale,
        ) = noise_scale_bounds

        if (
            minimum_noise_scale <= 0
            or maximum_noise_scale
            <= minimum_noise_scale
        ):
            raise ValueError(
                "noise_scale_bounds must contain "
                "positive increasing values."
            )

        local_noise_values = np.clip(
            local_noise_values,
            (
                global_noise_sigma
                * minimum_noise_scale
            ),
            (
                global_noise_sigma
                * maximum_noise_scale
            ),
        )

 
    local_noise_trace = np.interp(
        time,
        window_centers,
        local_noise_values,
    )

    local_center_trace = np.interp(
        time,
        window_centers,
        local_centers,
    )

    centered_filtered = (
        filtered_signal
        - local_center_trace
    )

 
    lower_threshold_trace = (
        threshold_factor
        * local_noise_trace
    )

    if upper_threshold_factor is None:
        upper_threshold_trace = None

    else:
        upper_threshold_trace = (
            upper_threshold_factor
            * local_noise_trace
        )

    if prominence_factor is None:
        prominence_trace = None

    else:
        prominence_trace = (
            prominence_factor
            * local_noise_trace
        )

 # polarity for detection 
    if polarity == "positive":
        detection_signal = (
            centered_filtered.copy()
        )

    elif polarity == "negative":
        detection_signal = (
            -centered_filtered
        )

    else:
        detection_signal = np.abs(
            centered_filtered
        )

    detection_signal[
        ~valid_samples
    ] = -np.inf

    minimum_distance_samples = max(
        1,
        int(
            sampling_rate
            * shadow_period_ms
            / 1000
        ),
    )

    height_range = (
        lower_threshold_trace,
        upper_threshold_trace,
    )

    if prominence_trace is None:
        prominence_range = None

    else:
        prominence_range = (
            prominence_trace,
            None,
        )

    spike_indices, properties = find_peaks(
        detection_signal,
        height=height_range,
        distance=minimum_distance_samples,
        prominence=prominence_range,
    )


    spike_times = time[
        spike_indices
    ]

    signed_amplitudes = centered_filtered[
        spike_indices
    ]

    absolute_amplitudes = np.abs(
        signed_amplitudes
    )

    local_noise_at_spikes = (
        local_noise_trace[
            spike_indices
        ]
    )

    amplitudes_in_noise_units = (
        absolute_amplitudes
        / local_noise_at_spikes
    )

    lower_threshold_at_spikes = (
        lower_threshold_trace[
            spike_indices
        ]
    )

    if upper_threshold_trace is None:
        upper_threshold_at_spikes = np.full(
            len(spike_indices),
            np.nan,
        )

    else:
        upper_threshold_at_spikes = (
            upper_threshold_trace[
                spike_indices
            ]
        )

    spike_polarities = np.where(
        signed_amplitudes >= 0,
        "positive",
        "negative",
    )

    spikes_df = pd.DataFrame({
        "spike_number": np.arange(
            1,
            len(spike_indices) + 1,
        ),
        "sample_index": spike_indices,
        "time_seconds": spike_times,
        "signed_amplitude": signed_amplitudes,
        "absolute_amplitude": absolute_amplitudes,
        "local_noise_sigma": local_noise_at_spikes,
        "lower_threshold": (
            lower_threshold_at_spikes
        ),
        "upper_threshold": (
            upper_threshold_at_spikes
        ),
        "amplitude_noise_units": (
            amplitudes_in_noise_units
        ),
        "polarity": spike_polarities,
        "polarity_mode": polarity,
    })

    noise_windows_df = pd.DataFrame({
        "window_center_seconds": (
            window_centers
        ),
        "valid_samples": valid_counts,
        "local_center": local_centers,
        "raw_local_noise_sigma": (
            raw_local_noise_values
        ),
        "used_local_noise_sigma": (
            local_noise_values
        ),
        "relative_to_global": (
            local_noise_values
            / global_noise_sigma
        ),
    })


    return {
        "time": time,
        "filtered_signal": filtered_signal,
        "centered_filtered_signal": (
            centered_filtered
        ),
        "artifact_mask": artifact_mask,
        "valid_samples": valid_samples,
        "sampling_rate": sampling_rate,
        "polarity_mode": polarity,

        # Global diagnostics
        "noise_sigma": global_noise_sigma,
        "global_noise_sigma": (
            global_noise_sigma
        ),
        "global_center": global_center,

        # Adaptive diagnostics
        "local_noise_trace": (
            local_noise_trace
        ),
        "local_center_trace": (
            local_center_trace
        ),
        "noise_windows_df": (
            noise_windows_df
        ),
        "adaptive_window_s": (
            adaptive_window_s
        ),
        "adaptive_step_s": (
            adaptive_step_s
        ),

        # Thresholds
        "threshold_factor": (
            threshold_factor
        ),
        "upper_threshold_factor": (
            upper_threshold_factor
        ),
        "prominence_factor": (
            prominence_factor
        ),
        "lower_threshold_trace": (
            lower_threshold_trace
        ),
        "upper_threshold_trace": (
            upper_threshold_trace
        ),
        "prominence_trace": (
            prominence_trace
        ),

        # Compatibility with older code
        "threshold": np.median(
            lower_threshold_trace[
                valid_samples
            ]
        ),
        "lower_threshold": np.median(
            lower_threshold_trace[
                valid_samples
            ]
        ),
        "upper_threshold": (
            None
            if upper_threshold_trace is None
            else np.median(
                upper_threshold_trace[
                    valid_samples
                ]
            )
        ),

        # Spike detections
        "shadow_period_ms": (
            shadow_period_ms
        ),
        "spike_indices": spike_indices,
        "spike_times": spike_times,
        "signed_amplitudes": (
            signed_amplitudes
        ),
        "absolute_amplitudes": (
            absolute_amplitudes
        ),
        "amplitudes_in_noise_units": (
            amplitudes_in_noise_units
        ),
        "spike_polarities": (
            spike_polarities
        ),
        "properties": properties,
        "spikes_df": spikes_df,
    }




def plot_adaptive_spike_detection(
    spike_result,
    start_time,
    duration=30.0,
    title=None,
):
    """
    Plot the signal, polarity-appropriate adaptive thresholds,
    detected spikes, artifact exclusions, and local noise.
    """

    time = np.asarray(
        spike_result["time"],
        dtype=float,
    )

    signal = np.asarray(
        spike_result[
            "centered_filtered_signal"
        ],
        dtype=float,
    )

    lower = np.asarray(
        spike_result[
            "lower_threshold_trace"
        ],
        dtype=float,
    )

    upper = spike_result[
        "upper_threshold_trace"
    ]

    artifact_mask = np.asarray(
        spike_result["artifact_mask"],
        dtype=bool,
    )

    local_noise = np.asarray(
        spike_result[
            "local_noise_trace"
        ],
        dtype=float,
    )

    spike_indices = np.asarray(
        spike_result["spike_indices"],
        dtype=int,
    )

    spike_times = np.asarray(
        spike_result["spike_times"],
        dtype=float,
    )

    polarity_mode = spike_result.get(
        "polarity_mode",
        "both",
    )

    stop_time = (
        start_time + duration
    )

    view_mask = (
        (time >= start_time)
        & (time <= stop_time)
    )

    if not np.any(view_mask):
        raise ValueError(
            "The requested plot interval does not "
            "overlap the recording."
        )

    visible_spike_mask = (
        (spike_times >= start_time)
        & (spike_times <= stop_time)
    )

    visible_spike_indices = spike_indices[
        visible_spike_mask
    ]

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(16, 8),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1]
        },
    )


    axes[0].plot(
        time[view_mask],
        signal[view_mask],
        color="black",
        linewidth=0.6,
        label="Filtered signal",
    )


    if polarity_mode in {
        "positive",
        "both",
    }:
        axes[0].plot(
            time[view_mask],
            lower[view_mask],
            color="blue",
            linestyle="--",
            linewidth=1,
            label="Positive lower threshold",
        )

    if polarity_mode in {
        "negative",
        "both",
    }:
        axes[0].plot(
            time[view_mask],
            -lower[view_mask],
            color="blue",
            linestyle="--",
            linewidth=1,
            label="Negative lower threshold",
        )

    if upper is not None:
        upper = np.asarray(
            upper,
            dtype=float,
        )

        if polarity_mode in {
            "positive",
            "both",
        }:
            axes[0].plot(
                time[view_mask],
                upper[view_mask],
                color="purple",
                linestyle=":",
                linewidth=1,
                label="Positive upper threshold",
            )

        if polarity_mode in {
            "negative",
            "both",
        }:
            axes[0].plot(
                time[view_mask],
                -upper[view_mask],
                color="purple",
                linestyle=":",
                linewidth=1,
                label="Negative upper threshold",
            )


    axes[0].scatter(
        time[visible_spike_indices],
        signal[visible_spike_indices],
        color="red",
        s=22,
        facecolors="none",
        linewidths=1.3,
        zorder=4,
        label="Detected spikes",
    )

    time_in_view = time[
        view_mask
    ]

    artifacts_in_view = artifact_mask[
        view_mask
    ]

    if np.any(artifacts_in_view):
        axes[0].fill_between(
            time_in_view,
            0,
            1,
            where=artifacts_in_view,
            transform=(
                axes[0]
                .get_xaxis_transform()
            ),
            color="orange",
            alpha=0.25,
            label="Artifact exclusion",
        )

    axes[0].set_ylabel(
        "Filtered voltage"
    )

    axes[0].grid(
        alpha=0.2
    )

    axes[0].legend(
        loc="upper right"
    )

    if title is None:
        title = (
            f"Adaptive spike detection "
            f"({polarity_mode} polarity)"
        )

    axes[0].set_title(
        title
    )


    axes[1].plot(
        time[view_mask],
        local_noise[view_mask],
        color="darkgreen",
        linewidth=1.5,
        label="Local noise estimate",
    )

    axes[1].axhline(
        spike_result[
            "global_noise_sigma"
        ],
        color="gray",
        linestyle="--",
        label="Global noise estimate",
    )

    axes[1].set_xlabel(
        "Time (seconds)"
    )

    axes[1].set_ylabel(
        "Local noise"
    )

    axes[1].grid(
        alpha=0.2
    )

    axes[1].legend(
        loc="upper right"
    )

    fig.tight_layout()
    plt.show()


In [ ]:

# Candidate sampling
number_of_time_bins = 20
number_of_amplitude_bins = 4
random_seed = 12345


# Signal stored inside processed_channels that should be shown
# for the reference channel.
reference_signal_key = "filtered_signal"

# Colors used in the two plots
candidate_trace_color = "#8f8f8f"
reference_trace_color = "#76a5d5"
candidate_marker_color = "#ffff00"
training_labels_path = Path(
    f"{channel_name}_manual_training_labels.csv"
)

if reference_channel_name is not None:
    if (
        reference_channel_name
        not in processed_channels
    ):
        raise KeyError(
            f"Reference channel "
            f"{reference_channel_name!r} was not found.\n"
            f"Available channels: "
            f"{list(processed_channels.keys())}"
        )

    reference_result = processed_channels[
        reference_channel_name
    ]

    if (
        reference_signal_key
        not in reference_result
    ):
        raise KeyError(
            f"{reference_signal_key!r} was not found "
            f"in processed_channels"
            f"[{reference_channel_name!r}].\n"
            f"Available keys: "
            f"{list(reference_result.keys())}"
        )

    reference_time = np.asarray(
        reference_result["time"],
        dtype=float,
    )

    reference_signal = np.asarray(
        reference_result[
            reference_signal_key
        ],
        dtype=float,
    )

    if (
        len(reference_time)
        != len(reference_signal)
    ):
        raise ValueError(
            "The reference-channel time and signal "
            "arrays do not have the same length."
        )

else:
    reference_time = None
    reference_signal = None


candidate_result = (
    detect_spikes_processed_channel_adaptive(
        result=processed_channels[
            channel_name
        ],

        # Intentionally permissive
        threshold_factor=8,
        upper_threshold_factor=None,
        prominence_factor=None,

        # Short shadow period to avoid losing close spikes
        shadow_period_ms=6,

        # SET POLARITY HERE
        polarity="positive",

        adaptive_window_s=30.0,
        adaptive_step_s=15.0,
        noise_quantile=0.25,
        minimum_valid_fraction=0.25,
        noise_scale_bounds=(0.25, 4.0),
    )
)

candidate_time = np.asarray(
    candidate_result["time"],
    dtype=float,
)

candidate_signal = np.asarray(
    candidate_result[
        "centered_filtered_signal"
    ],
    dtype=float,
)

candidate_indices = np.asarray(
    candidate_result[
        "spike_indices"
    ],
    dtype=int,
)

candidate_times = candidate_time[
    candidate_indices
]

local_noise_trace = np.asarray(
    candidate_result[
        "local_noise_trace"
    ],
    dtype=float,
)

candidate_amplitudes = np.abs(
    candidate_signal[
        candidate_indices
    ]
)

candidate_snr = (
    candidate_amplitudes
    / local_noise_trace[
        candidate_indices
    ]
)

print(
    f"Generated {len(candidate_indices):,} "
    f"permissive candidates for {channel_name}."
)


rng = np.random.default_rng(
    random_seed
)

time_edges = np.linspace(
    candidate_times.min(),
    candidate_times.max(),
    number_of_time_bins + 1,
)

time_bins = np.clip(
    np.digitize(
        candidate_times,
        time_edges[1:-1],
    ),
    0,
    number_of_time_bins - 1,
)

amplitude_edges = np.unique(
    np.quantile(
        candidate_snr,
        np.linspace(
            0,
            1,
            number_of_amplitude_bins + 1,
        ),
    )
)

if len(amplitude_edges) > 2:
    amplitude_bins = np.clip(
        np.digitize(
            candidate_snr,
            amplitude_edges[1:-1],
        ),
        0,
        len(amplitude_edges) - 2,
    )
else:
    amplitude_bins = np.zeros(
        len(candidate_snr),
        dtype=int,
    )

candidate_groups = {}

for position in range(
    len(candidate_indices)
):
    group_key = (
        int(time_bins[position]),
        int(amplitude_bins[position]),
    )

    candidate_groups.setdefault(
        group_key,
        [],
    ).append(position)

for group_positions in (
    candidate_groups.values()
):
    rng.shuffle(
        group_positions
    )

# Round-robin sampling across groups
candidate_order = []

group_keys = list(
    candidate_groups.keys()
)

rng.shuffle(group_keys)

groups_remaining = True

while groups_remaining:
    groups_remaining = False

    for group_key in group_keys:
        group_positions = (
            candidate_groups[
                group_key
            ]
        )

        if group_positions:
            candidate_order.append(
                group_positions.pop()
            )

            groups_remaining = True

candidate_order = np.asarray(
    candidate_order,
    dtype=int,
)


# ============================================================
# LOAD PREVIOUS LABELING PROGRESS
# ============================================================

if training_labels_path.exists():
    manual_labels_df = pd.read_csv(
        training_labels_path
    )

    print(
        f"Resuming from "
        f"{training_labels_path}: "
        f"{len(manual_labels_df)} labels."
    )

else:
    manual_labels_df = pd.DataFrame(
        columns=[
            "channel",
            "sample_index",
            "time_seconds",
            "manual_label",
            "candidate_snr",
        ]
    )

class CandidateLabeler(
    QtWidgets.QMainWindow
):
    def __init__(self):
        super().__init__()

        self.order_position = 0
        self.current_candidate_position = None
        self.history = []

        self.setWindowTitle(
            f"{channel_name}: candidate labeling"
        )

        # Slightly taller than before because there are
        # now two plots.
        self.resize(
            1450,
            950,
        )

        self.build_interface()
        self.install_shortcuts()
        self.advance_to_next_unlabeled()

    def build_interface(self):
        central = QtWidgets.QWidget()
        self.setCentralWidget(central)

        layout = QtWidgets.QVBoxLayout(
            central
        )

        self.status_label = (
            QtWidgets.QLabel()
        )

        self.status_label.setStyleSheet(
            "font-size: 16px; font-weight: bold;"
        )

        layout.addWidget(
            self.status_label
        )

   
        if hasattr(QtCore.Qt, "Orientation"):
            vertical_orientation = (
                QtCore.Qt.Orientation.Vertical
            )
        else:
            vertical_orientation = (
                QtCore.Qt.Vertical
            )

        self.plot_splitter = (
            QtWidgets.QSplitter(
                vertical_orientation
            )
        )

    
        self.plot = pg.PlotWidget()

        self.plot.showGrid(
            x=True,
            y=True,
            alpha=0.2,
        )

        self.plot.setLabel(
            "left",
            f"{channel_name} filtered voltage",
        )

        self.plot.setTitle(
            f"Candidate channel: {channel_name}"
        )

        
        self.plot.getAxis(
            "bottom"
        ).enableAutoSIPrefix(
            False
        )

        self.plot.getAxis(
            "left"
        ).enableAutoSIPrefix(
            False
        )

        self.plot_splitter.addWidget(
            self.plot
        )

      
        if reference_channel_name is not None:
            self.reference_plot = (
                pg.PlotWidget()
            )

            self.reference_plot.showGrid(
                x=True,
                y=True,
                alpha=0.2,
            )

            self.reference_plot.setLabel(
                "bottom",
                "Time (seconds)",
            )

            self.reference_plot.setLabel(
                "left",
                (
                    f"{reference_channel_name} "
                    f"{reference_signal_key}"
                ),
            )

            self.reference_plot.setTitle(
                f"Reference channel: "
                f"{reference_channel_name}"
            )

         
            self.reference_plot.setXLink(
                self.plot
            )

            self.reference_plot.getAxis(
                "bottom"
            ).enableAutoSIPrefix(
                False
            )

            self.reference_plot.getAxis(
                "left"
            ).enableAutoSIPrefix(
                False
            )

       
            self.plot.getAxis(
                "bottom"
            ).setStyle(
                showValues=False
            )

            self.plot_splitter.addWidget(
                self.reference_plot
            )

           
            self.plot_splitter.setStretchFactor(
                0,
                2,
            )

            self.plot_splitter.setStretchFactor(
                1,
                1,
            )

        else:
            self.reference_plot = None

            self.plot.setLabel(
                "bottom",
                "Time (seconds)",
            )

        layout.addWidget(
            self.plot_splitter,
            stretch=1,
        )

        button_layout = (
            QtWidgets.QHBoxLayout()
        )

        self.label_buttons = {}

        button_colors = [
            "#4daf4a",
            "#e41a1c",
            "#377eb8",
            "#984ea3",
        ]

        for index, label in enumerate(
            label_targets
        ):
            button = (
                QtWidgets.QPushButton(
                    f"{label} "
                    f"[{index + 1}]"
                )
            )

            button.setMinimumHeight(
                55
            )

            button.setStyleSheet(
                f"font-size: 16px; "
                f"background-color: "
                f"{button_colors[index % len(button_colors)]};"
            )

            button.clicked.connect(
                lambda checked=False, value=label:
                self.assign_label(value)
            )

            button_layout.addWidget(
                button
            )

            self.label_buttons[
                label
            ] = button

        self.skip_button = (
            QtWidgets.QPushButton(
                "Skip [S]"
            )
        )

        self.skip_button.setMinimumHeight(
            55
        )

        self.skip_button.clicked.connect(
            self.skip_candidate
        )

        button_layout.addWidget(
            self.skip_button
        )

        self.undo_button = (
            QtWidgets.QPushButton(
                "Undo [U]"
            )
        )

        self.undo_button.setMinimumHeight(
            55
        )

        self.undo_button.clicked.connect(
            self.undo_last
        )

        button_layout.addWidget(
            self.undo_button
        )

        self.save_button = (
            QtWidgets.QPushButton(
                "Save"
            )
        )

        self.save_button.setMinimumHeight(
            55
        )

        self.save_button.clicked.connect(
            self.save_labels
        )

        button_layout.addWidget(
            self.save_button
        )

        layout.addLayout(
            button_layout
        )

    def install_shortcuts(self):
        for index, label in enumerate(
            label_targets
        ):
            shortcut = QtGui.QShortcut(
                QtGui.QKeySequence(
                    str(index + 1)
                ),
                self,
            )

            shortcut.activated.connect(
                lambda value=label:
                self.assign_label(value)
            )

            # Preserve shortcut reference
            setattr(
                self,
                f"_shortcut_{index}",
                shortcut,
            )

        self.skip_shortcut = (
            QtGui.QShortcut(
                QtGui.QKeySequence("S"),
                self,
            )
        )

        self.skip_shortcut.activated.connect(
            self.skip_candidate
        )

        self.undo_shortcut = (
            QtGui.QShortcut(
                QtGui.QKeySequence("U"),
                self,
            )
        )

        self.undo_shortcut.activated.connect(
            self.undo_last
        )

    def current_counts(self):
        counts = {}

        for label in label_targets:
            counts[label] = int(
                np.sum(
                    manual_labels_df[
                        "manual_label"
                    ]
                    == label
                )
            )

        return counts

    def targets_complete(self):
        counts = self.current_counts()

        return all(
            counts[label]
            >= target
            for label, target
            in label_targets.items()
        )

    def update_buttons(self):
        counts = self.current_counts()

        for label, target in (
            label_targets.items()
        ):
            completed = (
                counts[label] >= target
            )

            self.label_buttons[
                label
            ].setEnabled(
                not completed
            )

            self.label_buttons[
                label
            ].setText(
                f"{label}: "
                f"{counts[label]}/{target}"
            )

    def candidate_already_labeled(
        self,
        candidate_position,
    ):
        sample_index = int(
            candidate_indices[
                candidate_position
            ]
        )

        return bool(
            np.any(
                pd.to_numeric(
                    manual_labels_df[
                        "sample_index"
                    ],
                    errors="coerce",
                )
                == sample_index
            )
        )

    def advance_to_next_unlabeled(self):
        if self.targets_complete():
            self.save_labels()

            self.status_label.setText(
                "Labeling targets complete. "
                "You can close this window and run "
                "the classifier block."
            )

            self.update_buttons()
            return

        while (
            self.order_position
            < len(candidate_order)
        ):
            candidate_position = int(
                candidate_order[
                    self.order_position
                ]
            )

            self.order_position += 1

            if not self.candidate_already_labeled(
                candidate_position
            ):
                self.current_candidate_position = (
                    candidate_position
                )

                self.show_current_candidate()
                return

        self.status_label.setText(
            "No additional unlabeled candidates remain."
        )

    @staticmethod
    def set_robust_y_range(
        plot_widget,
        signal_values,
    ):
        """
        Set an independent Y-axis range using the visible
        signal values in one plot.
        """

        finite_values = np.asarray(
            signal_values,
            dtype=float,
        )

        finite_values = finite_values[
            np.isfinite(finite_values)
        ]

        if len(finite_values) == 0:
            return

        low, high = np.percentile(
            finite_values,
            [0.5, 99.5],
        )

        if high <= low:
            center = float(
                np.median(finite_values)
            )

            half_range = max(
                abs(center) * 0.05,
                1e-12,
            )

            low = center - half_range
            high = center + half_range

        margin = max(
            (high - low) * 0.15,
            1e-12,
        )

        plot_widget.setYRange(
            low - margin,
            high + margin,
            padding=0,
        )

    def show_current_candidate(self):
        position = (
            self.current_candidate_position
        )

        spike_time = float(
            candidate_times[
                position
            ]
        )

        spike_index = int(
            candidate_indices[
                position
            ]
        )

        start_time = (
            spike_time
            - context_before_seconds
        )

        stop_time = (
            spike_time
            + context_after_seconds
        )

      
        left = np.searchsorted(
            candidate_time,
            start_time,
            side="left",
        )

        right = np.searchsorted(
            candidate_time,
            stop_time,
            side="right",
        )

        context_time = candidate_time[
            left:right
        ]

        context_signal = candidate_signal[
            left:right
        ]

        nearby_left = np.searchsorted(
            candidate_times,
            start_time,
            side="left",
        )

        nearby_right = np.searchsorted(
            candidate_times,
            stop_time,
            side="right",
        )

        nearby_times = candidate_times[
            nearby_left:nearby_right
        ]

        nearby_indices = candidate_indices[
            nearby_left:nearby_right
        ]

        nearby_values = candidate_signal[
            nearby_indices
        ]

        self.plot.clear()

        trace = self.plot.plot(
            context_time,
            context_signal,
            pen=pg.mkPen(
                candidate_trace_color,
                width=1,
            ),
        )

        trace.setDownsampling(
            auto=True,
            method="peak",
        )

        trace.setClipToView(
            True
        )

        self.plot.plot(
            nearby_times,
            nearby_values,
            pen=None,
            symbol="o",
            symbolSize=6,
            symbolPen=pg.mkPen(
                "#777777"
            ),
            symbolBrush=None,
        )

        current_value = candidate_signal[
            spike_index
        ]

        self.plot.plot(
            [spike_time],
            [current_value],
            pen=None,
            symbol="o",
            symbolSize=14,
            symbolPen=pg.mkPen(
                candidate_marker_color,
                width=3,
            ),
            symbolBrush=None,
        )

        marker_line = pg.InfiniteLine(
            pos=spike_time,
            angle=90,
            pen=pg.mkPen(
                candidate_marker_color,
                width=2,
            ),
        )

        self.plot.addItem(
            marker_line
        )

        self.plot.setXRange(
            start_time,
            stop_time,
            padding=0,
        )

        self.set_robust_y_range(
            self.plot,
            context_signal,
        )

      

        if self.reference_plot is not None:
            reference_left = np.searchsorted(
                reference_time,
                start_time,
                side="left",
            )

            reference_right = np.searchsorted(
                reference_time,
                stop_time,
                side="right",
            )

            visible_reference_time = (
                reference_time[
                    reference_left:
                    reference_right
                ]
            )

            visible_reference_signal = (
                reference_signal[
                    reference_left:
                    reference_right
                ]
            )

            self.reference_plot.clear()

            if len(visible_reference_time):
                reference_trace = (
                    self.reference_plot.plot(
                        visible_reference_time,
                        visible_reference_signal,
                        pen=pg.mkPen(
                            reference_trace_color,
                            width=1,
                        ),
                    )
                )

                reference_trace.setDownsampling(
                    auto=True,
                    method="peak",
                )

                reference_trace.setClipToView(
                    True
                )

                self.set_robust_y_range(
                    self.reference_plot,
                    visible_reference_signal,
                )

            
            reference_marker_line = (
                pg.InfiniteLine(
                    pos=spike_time,
                    angle=90,
                    pen=pg.mkPen(
                        candidate_marker_color,
                        width=2,
                    ),
                )
            )

            self.reference_plot.addItem(
                reference_marker_line
            )

            self.reference_plot.setXRange(
                start_time,
                stop_time,
                padding=0,
            )

       

        counts = self.current_counts()

        count_text = " | ".join(
            f"{label}: "
            f"{counts[label]}/"
            f"{label_targets[label]}"
            for label in label_targets
        )

        if reference_channel_name is None:
            reference_text = ""
        else:
            reference_text = (
                f" | Reference: "
                f"{reference_channel_name}"
            )

        self.status_label.setText(
            f"Candidate at {spike_time:.6f} s | "
            f"SNR={candidate_snr[position]:.2f}"
            f"{reference_text} | "
            f"{count_text}"
        )

        self.update_buttons()

    def assign_label(
        self,
        label,
    ):
        if (
            self.current_candidate_position
            is None
        ):
            return

        counts = self.current_counts()

        if (
            counts[label]
            >= label_targets[label]
        ):
            return

        position = (
            self.current_candidate_position
        )

        new_row = pd.DataFrame([{
            "channel": channel_name,
            "sample_index": int(
                candidate_indices[position]
            ),
            "time_seconds": float(
                candidate_times[position]
            ),
            "manual_label": label,
            "candidate_snr": float(
                candidate_snr[position]
            ),
        }])

        global manual_labels_df

        manual_labels_df = pd.concat(
            [
                manual_labels_df,
                new_row,
            ],
            ignore_index=True,
        )

        self.history.append(
            int(
                candidate_indices[
                    position
                ]
            )
        )

        self.save_labels()
        self.advance_to_next_unlabeled()

    def skip_candidate(
        self,
        *_,
    ):
        self.advance_to_next_unlabeled()

    def undo_last(
        self,
        *_,
    ):
        global manual_labels_df

        if not self.history:
            return

        sample_index = self.history.pop()

        matching_rows = (
            manual_labels_df.index[
                pd.to_numeric(
                    manual_labels_df[
                        "sample_index"
                    ],
                    errors="coerce",
                )
                == sample_index
            ]
        )

        if len(matching_rows):
            manual_labels_df = (
                manual_labels_df.drop(
                    matching_rows[-1]
                )
                .reset_index(drop=True)
            )

        self.save_labels()
        self.show_current_candidate()

    def save_labels(
        self,
        *_,
    ):
        manual_labels_df.to_csv(
            training_labels_path,
            index=False,
        )




pg.setConfigOptions(
    antialias=False,
)

app = QtWidgets.QApplication.instance()

if app is None:
    app = QtWidgets.QApplication(
        sys.argv
    )

if "candidate_labeler_window" in globals():
    try:
        candidate_labeler_window.close()
    except Exception:
        pass

candidate_labeler_window = (
    CandidateLabeler()
)

candidate_labeler_window.show()
candidate_labeler_window.raise_()
candidate_labeler_window.activateWindow()

#### 6.2A: Training Classifier

In [ ]:


waveform_before_ms = 2.0
waveform_after_ms = 3.0

# Predictions below this confidence remain uncertain
minimum_classification_probability = 0.85

# Time blocks used for validation
validation_block_seconds = 30.0

classification_output_path = Path(
    f"{channel_name}_supervised_candidates.csv"
)



manual_labels_df = pd.read_csv(
    training_labels_path
)

manual_labels_df[
    "sample_index"
] = pd.to_numeric(
    manual_labels_df["sample_index"],
    errors="coerce",
)

manual_labels_df = (
    manual_labels_df
    .dropna(subset=["sample_index"])
    .copy()
)

manual_labels_df[
    "sample_index"
] = (
    manual_labels_df[
        "sample_index"
    ].astype(int)
)

print(
    manual_labels_df[
        "manual_label"
    ].value_counts()
)




sampling_rate = float(
    candidate_result[
        "sampling_rate"
    ]
)

samples_before = max(
    1,
    int(
        sampling_rate
        * waveform_before_ms
        / 1000
    ),
)

samples_after = max(
    1,
    int(
        sampling_rate
        * waveform_after_ms
        / 1000
    ),
)

waveform_length = (
    samples_before
    + samples_after
    + 1
)


candidate_times_sorted = np.asarray(
    candidate_times,
    dtype=float,
)

preceding_isi = np.diff(
    candidate_times_sorted,
    prepend=np.nan,
)

following_isi = np.diff(
    candidate_times_sorted,
    append=np.nan,
)

preceding_isi = np.nan_to_num(
    preceding_isi,
    nan=10.0,
    posinf=10.0,
    neginf=10.0,
)

following_isi = np.nan_to_num(
    following_isi,
    nan=10.0,
    posinf=10.0,
    neginf=10.0,
)

preceding_isi = np.clip(
    preceding_isi,
    0,
    10,
)

following_isi = np.clip(
    following_isi,
    0,
    10,
)


def candidate_count_in_window(
    center_times,
    half_window,
):
    left = np.searchsorted(
        center_times,
        center_times - half_window,
        side="left",
    )

    right = np.searchsorted(
        center_times,
        center_times + half_window,
        side="right",
    )

    
    return (
        right - left - 1
    )


count_within_025 = (
    candidate_count_in_window(
        candidate_times_sorted,
        0.25,
    )
)

count_within_100 = (
    candidate_count_in_window(
        candidate_times_sorted,
        1.0,
    )
)


# Fast local absolute-signal averages using cumulative sums
finite_signal = np.where(
    np.isfinite(candidate_signal),
    candidate_signal,
    0.0,
)

valid_signal = np.isfinite(
    candidate_signal
).astype(float)

absolute_cumsum = np.concatenate([
    [0.0],
    np.cumsum(
        np.abs(finite_signal)
    ),
])

valid_cumsum = np.concatenate([
    [0.0],
    np.cumsum(valid_signal),
])


def local_absolute_mean(
    sample_index,
    half_window_seconds,
):
    half_samples = int(
        sampling_rate
        * half_window_seconds
    )

    left = max(
        0,
        sample_index - half_samples,
    )

    right = min(
        len(candidate_signal),
        sample_index + half_samples + 1,
    )

    total = (
        absolute_cumsum[right]
        - absolute_cumsum[left]
    )

    count = (
        valid_cumsum[right]
        - valid_cumsum[left]
    )

    if count <= 0:
        return np.nan

    return total / count


feature_rows = []
normalized_waveforms = []

for candidate_position, sample_index in enumerate(
    candidate_indices
):
    waveform_start = (
        sample_index - samples_before
    )

    waveform_stop = (
        sample_index
        + samples_after
        + 1
    )

    if (
        waveform_start < 0
        or waveform_stop
        > len(candidate_signal)
    ):
        continue

    waveform = candidate_signal[
        waveform_start:waveform_stop
    ].copy()

    if (
        len(waveform)
        != waveform_length
        or not np.all(
            np.isfinite(waveform)
        )
    ):
        continue

    baseline_count = max(
        1,
        samples_before // 2,
    )

    baseline = np.median(
        waveform[
            :baseline_count
        ]
    )

    waveform = waveform - baseline

    absolute_peak = np.max(
        np.abs(waveform)
    )

    if (
        not np.isfinite(absolute_peak)
        or absolute_peak <= 0
    ):
        continue

    normalized_waveform = (
        waveform / absolute_peak
    )

    peak_to_peak = np.ptp(
        waveform
    )

    waveform_energy = np.sum(
        waveform ** 2
    )

    positive_peak = np.max(
        waveform
    )

    negative_peak = np.min(
        waveform
    )

    local_noise = (
        local_noise_trace[
            sample_index
        ]
    )

    snr = (
        absolute_peak / local_noise
        if local_noise > 0
        else np.nan
    )

    feature_rows.append({
        "candidate_position": (
            candidate_position
        ),
        "sample_index": int(
            sample_index
        ),
        "time_seconds": float(
            candidate_times[
                candidate_position
            ]
        ),
        "absolute_peak": float(
            absolute_peak
        ),
        "positive_peak": float(
            positive_peak
        ),
        "negative_peak": float(
            negative_peak
        ),
        "peak_to_peak": float(
            peak_to_peak
        ),
        "waveform_energy": float(
            waveform_energy
        ),
        "local_noise_sigma": float(
            local_noise
        ),
        "snr": float(snr),
        "preceding_isi": float(
            preceding_isi[
                candidate_position
            ]
        ),
        "following_isi": float(
            following_isi[
                candidate_position
            ]
        ),
        "candidates_within_025s": int(
            count_within_025[
                candidate_position
            ]
        ),
        "candidates_within_1s": int(
            count_within_100[
                candidate_position
            ]
        ),
        "local_activity_025s": float(
            local_absolute_mean(
                int(sample_index),
                0.25,
            )
        ),
        "local_activity_1s": float(
            local_absolute_mean(
                int(sample_index),
                1.0,
            )
        ),
    })

    normalized_waveforms.append(
        normalized_waveform
    )


feature_df = pd.DataFrame(
    feature_rows
)

normalized_waveforms = np.asarray(
    normalized_waveforms,
    dtype=float,
)

for waveform_column in range(
    normalized_waveforms.shape[1]
):
    feature_df[
        f"waveform_{waveform_column:03d}"
    ] = normalized_waveforms[
        :,
        waveform_column,
    ]

print(
    f"Created features for "
    f"{len(feature_df):,} candidates."
)


#attatch manual labels
training_df = feature_df.merge(
    manual_labels_df[
        [
            "sample_index",
            "manual_label",
        ]
    ],
    on="sample_index",
    how="inner",
)

training_df = training_df[
    training_df[
        "manual_label"
    ].isin(
        label_targets.keys()
    )
].copy()

print("\nTraining examples:")

print(
    training_df[
        "manual_label"
    ].value_counts()
)



non_feature_columns = {
    "candidate_position",
    "sample_index",
    "time_seconds",
    "manual_label",
}

feature_columns = [
    column
    for column in feature_df.columns
    if column not in non_feature_columns
]

X = training_df[
    feature_columns
].to_numpy(dtype=float)

y = training_df[
    "manual_label"
].to_numpy()


feature_medians = np.nanmedian(
    X,
    axis=0,
)

invalid_rows, invalid_columns = np.where(
    ~np.isfinite(X)
)

X[
    invalid_rows,
    invalid_columns
] = feature_medians[
    invalid_columns
]



validation_groups = np.floor(
    (
        training_df[
            "time_seconds"
        ].to_numpy()
        - candidate_times.min()
    )
    / validation_block_seconds
).astype(int)

split_found = False

group_splitter = GroupShuffleSplit(
    n_splits=30,
    test_size=0.25,
    random_state=12345,
)

for train_indices, test_indices in (
    group_splitter.split(
        X,
        y,
        groups=validation_groups,
    )
):
    if (
        len(np.unique(y[train_indices]))
        == len(np.unique(y))
        and len(np.unique(y[test_indices]))
        == len(np.unique(y))
    ):
        split_found = True
        break

if not split_found:
    print(
        "Could not form a time-blocked split containing "
        "all labels. Falling back to a stratified random split."
    )

    (
        X_train,
        X_test,
        y_train,
        y_test,
    ) = train_test_split(
        X,
        y,
        test_size=0.25,
        stratify=y,
        random_state=12345,
    )

else:
    X_train = X[
        train_indices
    ]

    X_test = X[
        test_indices
    ]

    y_train = y[
        train_indices
    ]

    y_test = y[
        test_indices
    ]


# ============================================================
# TRAIN AND VALIDATE
# ============================================================

validation_model = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=12345,
)

validation_model.fit(
    X_train,
    y_train,
)

test_predictions = (
    validation_model.predict(
        X_test
    )
)

print("\nHeld-out validation:")

print(
    classification_report(
        y_test,
        test_predictions,
        zero_division=0,
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    cmap="Blues",
)

plt.title(
    "Held-out candidate classification"
)

plt.show()



classifier = RandomForestClassifier(
    n_estimators=750,
    class_weight="balanced",
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=12345,
)

classifier.fit(
    X,
    y,
)

trained_model_path = Path(
    f"{exp}_{signal}_classifier.joblib"
)

model_package = {
    "classifier": classifier,
    "feature_columns": feature_columns,
    "feature_medians": feature_medians,
}

joblib.dump(
    model_package,
    trained_model_path,
)

print(
    "Saved trained classifier:",
    trained_model_path.resolve(),
)

#classifying candidates 

X_all = feature_df[
    feature_columns
].to_numpy(dtype=float)

invalid_rows, invalid_columns = np.where(
    ~np.isfinite(X_all)
)

X_all[
    invalid_rows,
    invalid_columns
] = feature_medians[
    invalid_columns
]

predicted_labels = classifier.predict(
    X_all
)

predicted_probabilities = (
    classifier.predict_proba(
        X_all
    )
)

maximum_probabilities = np.max(
    predicted_probabilities,
    axis=1,
)

final_labels = np.where(
    maximum_probabilities
    >= minimum_classification_probability,
    predicted_labels,
    "Uncertain",
)

classified_candidates_df = (
    feature_df[
        [
            "candidate_position",
            "sample_index",
            "time_seconds",
            "absolute_peak",
            "snr",
        ]
    ].copy()
)

classified_candidates_df[
    "predicted_label"
] = predicted_labels

classified_candidates_df[
    "classification_probability"
] = maximum_probabilities

classified_candidates_df[
    "manual_label"
] = final_labels

classified_candidates_df[
    "channel"
] = channel_name

for class_index, class_name in enumerate(
    classifier.classes_
):
    classified_candidates_df[
        f"probability_{class_name}"
    ] = predicted_probabilities[
        :,
        class_index,
    ]

classified_candidates_df.to_csv(
    classification_output_path,
    index=False,
)

print(
    "\nFinal classifications:"
)

print(
    classified_candidates_df[
        "manual_label"
    ].value_counts()
)

print(
    "\nSaved:",
    classification_output_path.resolve()
)



plot_colors = {
    "PY": "#4daf4a",
    "Noise": "#e41a1c",
    "Uncertain": "#888888",
}

fig, axis = plt.subplots(
    figsize=(16, 6)
)

for label in sorted(
    classified_candidates_df[
        "manual_label"
    ].unique()
):
    label_rows = (
        classified_candidates_df[
            classified_candidates_df[
                "manual_label"
            ]
            == label
        ]
    )

    axis.scatter(
        label_rows[
            "time_seconds"
        ],
        label_rows[
            "absolute_peak"
        ],
        s=10,
        alpha=0.5,
        color=plot_colors.get(
            label,
            None,
        ),
        label=label,
    )

axis.set_xlabel(
    "Recording time (seconds)"
)

axis.set_ylabel(
    "Absolute spike amplitude"
)

axis.set_title(
    f"{channel_name}: supervised candidate classification"
)

axis.legend()
axis.grid(alpha=0.2)

plt.show()

#### 6.2B: Alternative pathway for subsequent data files [must edit first cell!]

In [ ]:
trained_model_path = Path(
    "SC0005_PY_classifier.joblib"
)

In [ ]:

waveform_before_ms = 2.0
waveform_after_ms = 3.0

# Predictions below this confidence remain uncertain
minimum_classification_probability = 0.85


validation_block_seconds = 30.0

classification_output_path = Path(
    f"{channel_name}_supervised_candidates.csv"
)


manual_labels_df = pd.read_csv(
    training_labels_path
)

manual_labels_df[
    "sample_index"
] = pd.to_numeric(
    manual_labels_df["sample_index"],
    errors="coerce",
)

manual_labels_df = (
    manual_labels_df
    .dropna(subset=["sample_index"])
    .copy()
)

manual_labels_df[
    "sample_index"
] = (
    manual_labels_df[
        "sample_index"
    ].astype(int)
)

print(
    manual_labels_df[
        "manual_label"
    ].value_counts()
)


sampling_rate = float(
    candidate_result[
        "sampling_rate"
    ]
)

samples_before = max(
    1,
    int(
        sampling_rate
        * waveform_before_ms
        / 1000
    ),
)

samples_after = max(
    1,
    int(
        sampling_rate
        * waveform_after_ms
        / 1000
    ),
)

waveform_length = (
    samples_before
    + samples_after
    + 1
)


candidate_times_sorted = np.asarray(
    candidate_times,
    dtype=float,
)

preceding_isi = np.diff(
    candidate_times_sorted,
    prepend=np.nan,
)

following_isi = np.diff(
    candidate_times_sorted,
    append=np.nan,
)

preceding_isi = np.nan_to_num(
    preceding_isi,
    nan=10.0,
    posinf=10.0,
    neginf=10.0,
)

following_isi = np.nan_to_num(
    following_isi,
    nan=10.0,
    posinf=10.0,
    neginf=10.0,
)

preceding_isi = np.clip(
    preceding_isi,
    0,
    10,
)

following_isi = np.clip(
    following_isi,
    0,
    10,
)


def candidate_count_in_window(
    center_times,
    half_window,
):
    left = np.searchsorted(
        center_times,
        center_times - half_window,
        side="left",
    )

    right = np.searchsorted(
        center_times,
        center_times + half_window,
        side="right",
    )

   
    return (
        right - left - 1
    )


count_within_025 = (
    candidate_count_in_window(
        candidate_times_sorted,
        0.25,
    )
)

count_within_100 = (
    candidate_count_in_window(
        candidate_times_sorted,
        1.0,
    )
)


finite_signal = np.where(
    np.isfinite(candidate_signal),
    candidate_signal,
    0.0,
)

valid_signal = np.isfinite(
    candidate_signal
).astype(float)

absolute_cumsum = np.concatenate([
    [0.0],
    np.cumsum(
        np.abs(finite_signal)
    ),
])

valid_cumsum = np.concatenate([
    [0.0],
    np.cumsum(valid_signal),
])


def local_absolute_mean(
    sample_index,
    half_window_seconds,
):
    half_samples = int(
        sampling_rate
        * half_window_seconds
    )

    left = max(
        0,
        sample_index - half_samples,
    )

    right = min(
        len(candidate_signal),
        sample_index + half_samples + 1,
    )

    total = (
        absolute_cumsum[right]
        - absolute_cumsum[left]
    )

    count = (
        valid_cumsum[right]
        - valid_cumsum[left]
    )

    if count <= 0:
        return np.nan

    return total / count


feature_rows = []
normalized_waveforms = []

for candidate_position, sample_index in enumerate(
    candidate_indices
):
    waveform_start = (
        sample_index - samples_before
    )

    waveform_stop = (
        sample_index
        + samples_after
        + 1
    )

    if (
        waveform_start < 0
        or waveform_stop
        > len(candidate_signal)
    ):
        continue

    waveform = candidate_signal[
        waveform_start:waveform_stop
    ].copy()

    if (
        len(waveform)
        != waveform_length
        or not np.all(
            np.isfinite(waveform)
        )
    ):
        continue

    baseline_count = max(
        1,
        samples_before // 2,
    )

    baseline = np.median(
        waveform[
            :baseline_count
        ]
    )

    waveform = waveform - baseline

    absolute_peak = np.max(
        np.abs(waveform)
    )

    if (
        not np.isfinite(absolute_peak)
        or absolute_peak <= 0
    ):
        continue

    normalized_waveform = (
        waveform / absolute_peak
    )

    peak_to_peak = np.ptp(
        waveform
    )

    waveform_energy = np.sum(
        waveform ** 2
    )

    positive_peak = np.max(
        waveform
    )

    negative_peak = np.min(
        waveform
    )

    local_noise = (
        local_noise_trace[
            sample_index
        ]
    )

    snr = (
        absolute_peak / local_noise
        if local_noise > 0
        else np.nan
    )

    feature_rows.append({
        "candidate_position": (
            candidate_position
        ),
        "sample_index": int(
            sample_index
        ),
        "time_seconds": float(
            candidate_times[
                candidate_position
            ]
        ),
        "absolute_peak": float(
            absolute_peak
        ),
        "positive_peak": float(
            positive_peak
        ),
        "negative_peak": float(
            negative_peak
        ),
        "peak_to_peak": float(
            peak_to_peak
        ),
        "waveform_energy": float(
            waveform_energy
        ),
        "local_noise_sigma": float(
            local_noise
        ),
        "snr": float(snr),
        "preceding_isi": float(
            preceding_isi[
                candidate_position
            ]
        ),
        "following_isi": float(
            following_isi[
                candidate_position
            ]
        ),
        "candidates_within_025s": int(
            count_within_025[
                candidate_position
            ]
        ),
        "candidates_within_1s": int(
            count_within_100[
                candidate_position
            ]
        ),
        "local_activity_025s": float(
            local_absolute_mean(
                int(sample_index),
                0.25,
            )
        ),
        "local_activity_1s": float(
            local_absolute_mean(
                int(sample_index),
                1.0,
            )
        ),
    })

    normalized_waveforms.append(
        normalized_waveform
    )


feature_df = pd.DataFrame(
    feature_rows
)

normalized_waveforms = np.asarray(
    normalized_waveforms,
    dtype=float,
)

for waveform_column in range(
    normalized_waveforms.shape[1]
):
    feature_df[
        f"waveform_{waveform_column:03d}"
    ] = normalized_waveforms[
        :,
        waveform_column,
    ]

print(
    f"Created features for "
    f"{len(feature_df):,} candidates."
)



In [ ]:



minimum_classification_probability = 0.85

classification_output_path = Path(
    f"{channel_name}_"
    f"transferred_classifier_candidates.csv"
)

model_package = joblib.load(
    trained_model_path
)

classifier = model_package[
    "classifier"
]

feature_columns = model_package[
    "feature_columns"
]

feature_medians = np.asarray(
    model_package[
        "feature_medians"
    ],
    dtype=float,
)

missing_feature_columns = [
    column
    for column in feature_columns
    if column not in feature_df.columns
]

if missing_feature_columns:
    raise KeyError(
        "Missing required feature columns: "
        f"{missing_feature_columns}"
    )

X_all = feature_df[
    feature_columns
].to_numpy(
    dtype=float
)

invalid_rows, invalid_columns = np.where(
    ~np.isfinite(X_all)
)

X_all[
    invalid_rows,
    invalid_columns
] = feature_medians[
    invalid_columns
]

predicted_labels = classifier.predict(
    X_all
)

predicted_probabilities = (
    classifier.predict_proba(
        X_all
    )
)

maximum_probabilities = np.max(
    predicted_probabilities,
    axis=1,
)

final_labels = np.where(
    maximum_probabilities
    >= minimum_classification_probability,
    predicted_labels,
    "Uncertain",
)

classified_candidates_df = (
    feature_df[
        [
            "candidate_position",
            "sample_index",
            "time_seconds",
            "absolute_peak",
            "snr",
        ]
    ]
    .copy()
)

classified_candidates_df[
    "predicted_label"
] = predicted_labels

classified_candidates_df[
    "classification_probability"
] = maximum_probabilities

classified_candidates_df[
    "manual_label"
] = final_labels

classified_candidates_df[
    "channel"
] = channel_name

for class_index, class_name in enumerate(
    classifier.classes_
):
    classified_candidates_df[
        f"probability_{class_name}"
    ] = predicted_probabilities[
        :,
        class_index
    ]

classified_candidates_df.to_csv(
    classification_output_path,
    index=False,
)

print(
    "\nClassifications:"
)

print(
    classified_candidates_df[
        "manual_label"
    ].value_counts()
)

print(
    "\nSaved:",
    classification_output_path.resolve(),
)

display(
    classified_candidates_df.head()
)

### Step 7: find waveform, temporal, and rhythmic features 

In [ ]:
#User settings 

# Channel being classified
classification_channel_name = channel_name


# Rhythmic reference channel
reference_channel_name = "Green"

# Signal from processed_channels used for reference activity
reference_signal_key = "filtered_signal"

# Waveform extracted around every permissive candidate
waveform_before_ms = 5.0
waveform_after_ms = 5.0

# Threshold for identifying reference bursts.
#change this if reference busts are not being piked up from the envelope (lower)
# Increase this if noise is being counted as reference activity.
reference_burst_threshold_factor = 2

#Change this if you have a very slow freq. bursting 
# Join pieces of a burst separated by a short gap
maximum_reference_burst_gap_s = 0.150
# Reference activity lasting longer than this is treated as
# continuously active and therefore unreliable for burst/phase rules.
maximum_reliable_reference_burst_duration_s = 10.0

# Extend unreliable regions slightly on both sides so uncertain
# transitions are not treated as valid reference timing.
unreliable_reference_padding_s = 0.25


#### 7.1: find waveform, temporal, and rhythmic features 

In [ ]:


# Local candidate-density window
local_context_seconds = 0.250

# Reference-channel envelope smoothing
reference_envelope_smoothing_ms = 40.0


# Remove very short reference activity periods
minimum_reference_burst_duration_s = 0.100


# Number of seconds shown in the diagnostic reference plot
reference_diagnostic_seconds = 25.0

# Starting point of diagnostic plot
reference_diagnostic_start_s = float(
    candidate_times[0]
)
#reference_diagnostic_start_s = 7855

# ============================================================
# TARGET-CHANNEL INFORMATION
# ============================================================

candidate_time = np.asarray(
    candidate_result["time"],
    dtype=float,
)

candidate_signal = np.asarray(
    candidate_result[
        "centered_filtered_signal"
    ],
    dtype=float,
)

candidate_indices = np.asarray(
    candidate_result["spike_indices"],
    dtype=int,
)

candidate_times = candidate_time[
    candidate_indices
]

if len(candidate_time) < 2:
    raise ValueError(
        "Not enough target-channel samples."
    )

target_sampling_rate = 1.0 / np.median(
    np.diff(candidate_time)
)

samples_before = max(
    1,
    int(
        round(
            waveform_before_ms
            * target_sampling_rate
            / 1000.0
        )
    ),
)

samples_after = max(
    1,
    int(
        round(
            waveform_after_ms
            * target_sampling_rate
            / 1000.0
        )
    ),
)

waveform_sample_count = (
    samples_before
    + samples_after
    + 1
)

print(
    f"Target sampling rate: "
    f"{target_sampling_rate:.2f} Hz"
)

print(
    f"Waveform contains "
    f"{waveform_sample_count} samples."
)




waveforms = np.full(
    (
        len(candidate_indices),
        waveform_sample_count,
    ),
    np.nan,
    dtype=float,
)

valid_waveform = np.zeros(
    len(candidate_indices),
    dtype=bool,
)

for position, sample_index in enumerate(
    candidate_indices
):
    start_index = (
        sample_index - samples_before
    )

    stop_index = (
        sample_index
        + samples_after
        + 1
    )

    if start_index < 0:
        continue

    if stop_index > len(candidate_signal):
        continue

    waveform = candidate_signal[
        start_index:stop_index
    ].copy()

    if len(waveform) != waveform_sample_count:
        continue

    if not np.all(
        np.isfinite(waveform)
    ):
        continue

    # Remove the pre-spike baseline
    baseline_sample_count = max(
        1,
        samples_before // 2,
    )

    baseline = np.median(
        waveform[
            :baseline_sample_count
        ]
    )

    waveform = waveform - baseline

    waveforms[position] = waveform
    valid_waveform[position] = True




normalized_waveforms = np.full_like(
    waveforms,
    np.nan,
)

waveform_norms = np.linalg.norm(
    np.nan_to_num(waveforms),
    axis=1,
)

normalizable = (
    valid_waveform
    & np.isfinite(waveform_norms)
    & (waveform_norms > 0)
)

normalized_waveforms[
    normalizable
] = (
    waveforms[normalizable]
    / waveform_norms[
        normalizable,
        np.newaxis,
    ]
)



peak_positive = np.nanmax(
    waveforms,
    axis=1,
)

peak_negative = np.nanmin(
    waveforms,
    axis=1,
)

absolute_peak = np.nanmax(
    np.abs(waveforms),
    axis=1,
)

peak_to_peak = (
    peak_positive
    - peak_negative
)

waveform_energy = np.nansum(
    waveforms**2,
    axis=1,
)

waveform_area = np.nansum(
    np.abs(waveforms),
    axis=1,
)

peak_sample_within_waveform = np.nanargmax(
    np.abs(
        np.nan_to_num(waveforms)
    ),
    axis=1,
)

peak_offset_ms = (
    (
        peak_sample_within_waveform
        - samples_before
    )
    / target_sampling_rate
    * 1000.0
)



# Candidate indices from find_peaks should already be sorted,
# but explicitly sort them to guarantee correct ISIs.
candidate_sort_order = np.argsort(
    candidate_times
)

sorted_candidate_times = candidate_times[
    candidate_sort_order
]

previous_isi_s_sorted = np.full(
    len(candidate_times),
    np.nan,
)

next_isi_s_sorted = np.full(
    len(candidate_times),
    np.nan,
)

if len(candidate_times) > 1:
    candidate_intervals = np.diff(
        sorted_candidate_times
    )

    previous_isi_s_sorted[1:] = (
        candidate_intervals
    )

    next_isi_s_sorted[:-1] = (
        candidate_intervals
    )

previous_isi_s = np.full(
    len(candidate_times),
    np.nan,
)

next_isi_s = np.full(
    len(candidate_times),
    np.nan,
)

previous_isi_s[
    candidate_sort_order
] = previous_isi_s_sorted

next_isi_s[
    candidate_sort_order
] = next_isi_s_sorted

local_candidate_count_sorted = np.zeros(
    len(candidate_times),
    dtype=int,
)

for sorted_position, event_time in enumerate(
    sorted_candidate_times
):
    left_position = np.searchsorted(
        sorted_candidate_times,
        event_time - local_context_seconds,
        side="left",
    )

    right_position = np.searchsorted(
        sorted_candidate_times,
        event_time + local_context_seconds,
        side="right",
    )

    
    local_candidate_count_sorted[
        sorted_position
    ] = max(
        0,
        right_position
        - left_position
        - 1,
    )

local_candidate_count = np.zeros(
    len(candidate_times),
    dtype=int,
)

local_candidate_count[
    candidate_sort_order
] = local_candidate_count_sorted

local_candidate_rate_hz = (
    local_candidate_count
    / (2.0 * local_context_seconds)
)



if (
    reference_channel_name
    not in processed_channels
):
    raise KeyError(
        f"{reference_channel_name!r} was not found. "
        f"Available channels: "
        f"{list(processed_channels.keys())}"
    )

reference_result = processed_channels[
    reference_channel_name
]

reference_time = np.asarray(
    reference_result["time"],
    dtype=float,
)

reference_signal = np.asarray(
    reference_result[
        reference_signal_key
    ],
    dtype=float,
)

if len(reference_time) != len(
    reference_signal
):
    raise ValueError(
        "Reference time and signal lengths differ."
    )

if len(reference_time) < 2:
    raise ValueError(
        "Not enough reference-channel samples."
    )

reference_sampling_rate = (
    1.0
    / np.median(
        np.diff(reference_time)
    )
)

finite_reference = np.isfinite(
    reference_signal
)

if np.sum(finite_reference) < 10:
    raise ValueError(
        "Too few finite reference-channel samples."
    )

reference_center = np.nanmedian(
    reference_signal
)

reference_filled = (
    reference_signal.copy()
)

reference_filled[
    ~finite_reference
] = reference_center

reference_centered = (
    reference_filled
    - reference_center
)

smoothing_sigma_samples = max(
    1,
    int(
        round(
            reference_envelope_smoothing_ms
            * reference_sampling_rate
            / 1000.0
        )
    ),
)

# A smoothed rectified signal is computationally cheaper
# than applying a Hilbert transform to an entire long recording.
reference_envelope = gaussian_filter1d(
    np.abs(reference_centered),
    sigma=smoothing_sigma_samples,
    mode="nearest",
)



reference_envelope_center = np.median(
    reference_envelope
)

reference_envelope_sigma = (
    np.median(
        np.abs(
            reference_envelope
            - reference_envelope_center
        )
    )
    / 0.6745
)

if (
    not np.isfinite(
        reference_envelope_sigma
    )
    or reference_envelope_sigma <= 0
):
    reference_envelope_sigma = np.std(
        reference_envelope
    )

reference_burst_threshold = (
    reference_envelope_center
    + reference_burst_threshold_factor
    * reference_envelope_sigma
)

reference_active = (
    reference_envelope
    >= reference_burst_threshold
)



minimum_burst_samples = max(
    1,
    int(
        round(
            minimum_reference_burst_duration_s
            * reference_sampling_rate
        )
    ),
)

maximum_gap_samples = max(
    1,
    int(
        round(
            maximum_reference_burst_gap_s
            * reference_sampling_rate
        )
    ),
)


reference_active = binary_closing(
    reference_active,
    structure=np.ones(
        maximum_gap_samples,
        dtype=bool,
    ),
)


reference_active = binary_opening(
    reference_active,
    structure=np.ones(
        minimum_burst_samples,
        dtype=bool,
    ),
)




reference_transitions = np.diff(
    reference_active.astype(int),
    prepend=0,
    append=0,
)

reference_onset_indices = np.flatnonzero(
    reference_transitions == 1
)

reference_offset_indices = np.flatnonzero(
    reference_transitions == -1
)

reference_burst_onsets = reference_time[
    np.clip(
        reference_onset_indices,
        0,
        len(reference_time) - 1,
    )
]

reference_burst_offsets = reference_time[
    np.clip(
        reference_offset_indices - 1,
        0,
        len(reference_time) - 1,
    )
]

reference_burst_durations = (
    reference_burst_offsets
    - reference_burst_onsets
)



reference_burst_is_reliable = (
    np.isfinite(
        reference_burst_durations
    )
    & (
        reference_burst_durations
        <= maximum_reliable_reference_burst_duration_s
    )
)

reference_burst_is_unreliable = (
    ~reference_burst_is_reliable
)

unreliable_reference_mask = np.zeros(
    len(reference_time),
    dtype=bool,
)

unreliable_padding_samples = max(
    0,
    int(
        round(
            unreliable_reference_padding_s
            * reference_sampling_rate
        )
    ),
)

for onset_index, offset_index, is_unreliable in zip(
    reference_onset_indices,
    reference_offset_indices,
    reference_burst_is_unreliable,
):
    if not is_unreliable:
        continue

    padded_start = max(
        0,
        int(onset_index)
        - unreliable_padding_samples,
    )

    padded_stop = min(
        len(reference_time),
        int(offset_index)
        + unreliable_padding_samples,
    )

    unreliable_reference_mask[
        padded_start:padded_stop
    ] = True

reference_reliable_mask = (
    ~unreliable_reference_mask
)

reliable_reference_burst_onsets = (
    reference_burst_onsets[
        reference_burst_is_reliable
    ]
)

reliable_reference_burst_offsets = (
    reference_burst_offsets[
        reference_burst_is_reliable
    ]
)

reliable_reference_burst_durations = (
    reference_burst_durations[
        reference_burst_is_reliable
    ]
)

print(
    "Reliable reference bursts:",
    len(
        reliable_reference_burst_onsets
    ),
)

print(
    "Continuously active/unreliable "
    "reference periods:",
    int(
        np.sum(
            reference_burst_is_unreliable
        )
    ),
)



print(
    f"Reference channel: "
    f"{reference_channel_name}"
)

print(
    f"Reference sampling rate: "
    f"{reference_sampling_rate:.2f} Hz"
)

print(
    f"Reference bursts detected: "
    f"{len(reference_burst_onsets):,}"
)




candidate_reference_envelope = np.interp(
    candidate_times,
    reference_time,
    reference_envelope,
    left=np.nan,
    right=np.nan,
)

candidate_reference_z = (
    candidate_reference_envelope
    - reference_envelope_center
) / reference_envelope_sigma

candidate_reference_active = (
    np.interp(
        candidate_times,
        reference_time,
        reference_active.astype(float),
        left=0,
        right=0,
    )
    >= 0.5
)

candidate_reference_reliable = (
    np.interp(
        candidate_times,
        reference_time,
        reference_reliable_mask.astype(float),
        left=0,
        right=0,
    )
    >= 0.5
)

candidate_reference_unreliable = (
    ~candidate_reference_reliable
)

# Missing reference values tell the preprocessing pipeline not to
# treat continuous activity as a trustworthy rhythmic measurement.
candidate_reference_envelope[
    candidate_reference_unreliable
] = np.nan

candidate_reference_z[
    candidate_reference_unreliable
] = np.nan

candidate_reference_active_feature = (
    candidate_reference_active.astype(float)
)

candidate_reference_active_feature[
    candidate_reference_unreliable
] = np.nan




cycle_phase = np.full(
    len(candidate_times),
    np.nan,
)

time_from_previous_reference_onset = (
    np.full(
        len(candidate_times),
        np.nan,
    )
)

time_until_next_reference_onset = (
    np.full(
        len(candidate_times),
        np.nan,
    )
)

if len(reliable_reference_burst_onsets) >= 2:
    previous_onset_positions = (
        np.searchsorted(
            reliable_reference_burst_onsets,
            candidate_times,
            side="right",
        )
        - 1
    )

    next_onset_positions = (
        previous_onset_positions + 1
    )

    valid_cycle = (
        (previous_onset_positions >= 0)
        & (
            next_onset_positions
            < len(reliable_reference_burst_onsets)
        )
    )

    previous_onsets = reliable_reference_burst_onsets[
        previous_onset_positions[
            valid_cycle
        ]
    ]

    next_onsets = reliable_reference_burst_onsets[
        next_onset_positions[
            valid_cycle
        ]
    ]

    cycle_durations = (
        next_onsets
        - previous_onsets
    )

    interval_is_reliable = np.ones(
        len(cycle_durations),
        dtype=bool,
    )

    for interval_position, (
        previous_onset,
        next_onset,
    ) in enumerate(
        zip(
            previous_onsets,
            next_onsets,
        )
    ):
        interval_left = np.searchsorted(
            reference_time,
            previous_onset,
            side="left",
        )

        interval_right = np.searchsorted(
            reference_time,
            next_onset,
            side="right",
        )

        if np.any(
            unreliable_reference_mask[
                interval_left:interval_right
            ]
        ):
            interval_is_reliable[
                interval_position
            ] = False

    positive_cycle = (
        (cycle_durations > 0)
        & interval_is_reliable
        & candidate_reference_reliable[
            np.flatnonzero(valid_cycle)
        ]
    )

    valid_candidate_positions = np.flatnonzero(
        valid_cycle
    )

    valid_candidate_positions = (
        valid_candidate_positions[
            positive_cycle
        ]
    )

    previous_onsets = previous_onsets[
        positive_cycle
    ]

    next_onsets = next_onsets[
        positive_cycle
    ]

    cycle_durations = cycle_durations[
        positive_cycle
    ]

    cycle_phase[
        valid_candidate_positions
    ] = (
        (
            candidate_times[
                valid_candidate_positions
            ]
            - previous_onsets
        )
        / cycle_durations
    )

    time_from_previous_reference_onset[
        valid_candidate_positions
    ] = (
        candidate_times[
            valid_candidate_positions
        ]
        - previous_onsets
    )

    time_until_next_reference_onset[
        valid_candidate_positions
    ] = (
        next_onsets
        - candidate_times[
            valid_candidate_positions
        ]
    )


cycle_phase_sin = np.sin(
    2.0 * np.pi * cycle_phase
)

cycle_phase_cos = np.cos(
    2.0 * np.pi * cycle_phase
)




candidate_features_df = pd.DataFrame({
    "candidate_position": np.arange(
        len(candidate_indices)
    ),
    "sample_index": candidate_indices,
    "time_seconds": candidate_times,

    # Waveform amplitude/shape
    "positive_peak": peak_positive,
    "negative_peak": peak_negative,
    "absolute_peak": absolute_peak,
    "peak_to_peak": peak_to_peak,
    "waveform_energy": waveform_energy,
    "waveform_area": waveform_area,
    "peak_offset_ms": peak_offset_ms,

    # Candidate timing
    "previous_isi_ms": (
        previous_isi_s * 1000.0
    ),
    "next_isi_ms": (
        next_isi_s * 1000.0
    ),
    "local_candidate_count": (
        local_candidate_count
    ),
    "local_candidate_rate_hz": (
        local_candidate_rate_hz
    ),

    # Reference-channel context
    "reference_envelope": (
        candidate_reference_envelope
    ),
    "reference_envelope_z": (
        candidate_reference_z
    ),
    "reference_active": (
        candidate_reference_active_feature
    ),
    "reference_reliable": (
        candidate_reference_reliable.astype(int)
    ),
    "time_from_reference_onset_s": (
        time_from_previous_reference_onset
    ),
    "time_until_reference_onset_s": (
        time_until_next_reference_onset
    ),
    "cycle_phase": cycle_phase,
    "cycle_phase_sin": cycle_phase_sin,
    "cycle_phase_cos": cycle_phase_cos,

    "valid_waveform": valid_waveform,
})

# Store normalized waveform samples as separate columns.
# PCA will be fitted to these columns in Block 2.
waveform_feature_columns = []

for waveform_sample in range(
    waveform_sample_count
):
    column_name = (
        f"waveform_{waveform_sample:03d}"
    )

    candidate_features_df[
        column_name
    ] = normalized_waveforms[
        :,
        waveform_sample
    ]

    waveform_feature_columns.append(
        column_name
    )


# extracted features 
context_feature_columns = [
    "positive_peak",
    "negative_peak",
    "absolute_peak",
    "peak_to_peak",
    "waveform_energy",
    "waveform_area",
    "peak_offset_ms",
    "previous_isi_ms",
    "next_isi_ms",
    "local_candidate_count",
    "local_candidate_rate_hz",
    "reference_envelope_z",
    "reference_active",
    "reference_reliable",
    "time_from_reference_onset_s",
    "time_until_reference_onset_s",
    "cycle_phase_sin",
    "cycle_phase_cos",
]

print(
    f"Created features for "
    f"{len(candidate_features_df):,} candidates."
)




diagnostic_stop_s = (
    reference_diagnostic_start_s
    + reference_diagnostic_seconds
)

diagnostic_mask = (
    (reference_time >= reference_diagnostic_start_s)
    & (reference_time <= diagnostic_stop_s)
)

fig, axes = plt.subplots(
    2,
    1,
    figsize=(15, 7),
    sharex=True,
)

axes[0].plot(
    reference_time[diagnostic_mask],
    reference_signal[diagnostic_mask],
    color="0.45",
    linewidth=0.7,
)

axes[0].set_ylabel(
    f"{reference_channel_name}\nsignal"
)

axes[0].set_title(
    "Check whether reference bursts were identified correctly"
)

axes[1].plot(
    reference_time[diagnostic_mask],
    reference_envelope[diagnostic_mask],
    color="tab:blue",
    linewidth=1,
    label="Reference envelope",
)

axes[1].axhline(
    reference_burst_threshold,
    color="red",
    linestyle="--",
    label="Burst threshold",
)

active_plot_mask = (
    diagnostic_mask
    & reference_active
)

axes[1].fill_between(
    reference_time[diagnostic_mask],
    0,
    reference_envelope[diagnostic_mask],
    where=reference_active[
        diagnostic_mask
    ],
    color="tab:green",
    alpha=0.25,
    label="Detected reference burst",
)

axes[1].fill_between(
    reference_time[diagnostic_mask],
    0,
    reference_envelope[diagnostic_mask],
    where=unreliable_reference_mask[
        diagnostic_mask
    ],
    color="tab:red",
    alpha=0.35,
    label="Reference unavailable: continuous activity",
)

axes[1].set_ylabel(
    "Envelope"
)

axes[1].set_xlabel(
    "Time (seconds)"
)

axes[1].legend(
    loc="upper right"
)

plt.tight_layout()
plt.show()

# Preserve all originally detected boundaries for inspection.
all_reference_burst_onsets = reference_burst_onsets.copy()
all_reference_burst_offsets = reference_burst_offsets.copy()
all_reference_burst_durations = reference_burst_durations.copy()

# Downstream code continues to use these established variable names,
# but now receives only reliable burst boundaries. Candidates inside
# continuously active periods therefore receive no reference burst ID,
# and retain the training-classifier result without reference filtering.
reference_burst_onsets = (
    reliable_reference_burst_onsets.copy()
)

reference_burst_offsets = (
    reliable_reference_burst_offsets.copy()
)

reference_burst_durations = (
    reliable_reference_burst_durations.copy()
)


#### 7.2A: training context aware classifier [must edit first cell!!]

In [ ]:
#user settings 

# Number of PCA components describing normalized waveform shape.
number_of_waveform_pcs = 5

# Number of time sections used for validation.
# Each fold tests on a different part of the recording.
number_of_validation_time_blocks = 5

# Random forest settings
number_of_trees = 500
minimum_samples_per_leaf = 2
classifier_random_seed = 12345




##### train and validate classifier 

In [ ]:


required_label_columns = {
    "sample_index",
    "manual_label",
}

missing_columns = (
    required_label_columns
    - set(manual_labels_df.columns)
)

if missing_columns:
    raise KeyError(
        f"manual_labels_df is missing: "
        f"{missing_columns}"
    )

training_labels_df = (
    manual_labels_df[
        [
            "sample_index",
            "manual_label",
        ]
    ]
    .copy()
)

training_labels_df[
    "sample_index"
] = pd.to_numeric(
    training_labels_df[
        "sample_index"
    ],
    errors="coerce",
)

training_labels_df = (
    training_labels_df
    .dropna(
        subset=[
            "sample_index",
            "manual_label",
        ]
    )
    .copy()
)

training_labels_df[
    "sample_index"
] = training_labels_df[
    "sample_index"
].astype(int)


training_labels_df = (
    training_labels_df
    .drop_duplicates(
        subset="sample_index",
        keep="last",
    )
)

labeled_features_df = (
    candidate_features_df.merge(
        training_labels_df,
        on="sample_index",
        how="inner",
    )
)

labeled_features_df = labeled_features_df[
    labeled_features_df[
        "valid_waveform"
    ]
].copy()

if len(labeled_features_df) == 0:
    raise ValueError(
        "No manually labeled candidates matched "
        "the candidate feature table."
    )

label_counts = (
    labeled_features_df[
        "manual_label"
    ]
    .value_counts()
)

print(
    "Training examples by class:"
)

display(
    label_counts.rename(
        "number_of_examples"
    )
)

if len(label_counts) < 2:
    raise ValueError(
        "At least two manual-label classes are required."
    )




maximum_possible_pcs = min(
    number_of_waveform_pcs,
    len(waveform_feature_columns),
    max(
        1,
        len(labeled_features_df) - 1,
    ),
)

waveform_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value=0.0,
        ),
    ),
    (
        "scaler",
        StandardScaler(),
    ),
    (
        "pca",
        PCA(
            n_components=maximum_possible_pcs,
            random_state=classifier_random_seed,
        ),
    ),
])

context_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
        ),
    ),
    (
        "scaler",
        StandardScaler(),
    ),
])

feature_preprocessor = ColumnTransformer([
    (
        "waveform_shape",
        waveform_pipeline,
        waveform_feature_columns,
    ),
    (
        "temporal_and_rhythm",
        context_pipeline,
        context_feature_columns,
    ),
])



classifier = RandomForestClassifier(
    n_estimators=number_of_trees,
    min_samples_leaf=(
        minimum_samples_per_leaf
    ),
    class_weight="balanced",
    random_state=classifier_random_seed,
    n_jobs=-1,
    max_features="sqrt",
)

context_classifier = Pipeline([
    (
        "features",
        feature_preprocessor,
    ),
    (
        "classifier",
        classifier,
    ),
])




X_labeled = labeled_features_df[
    waveform_feature_columns
    + context_feature_columns
]

y_labeled = labeled_features_df[
    "manual_label"
].astype(str)

number_of_groups = min(
    number_of_validation_time_blocks,
    len(labeled_features_df),
)

if number_of_groups < 2:
    raise ValueError(
        "Not enough labeled examples for validation."
    )

time_group_edges = np.linspace(
    labeled_features_df[
        "time_seconds"
    ].min(),
    labeled_features_df[
        "time_seconds"
    ].max(),
    number_of_groups + 1,
)

validation_groups = np.clip(
    np.digitize(
        labeled_features_df[
            "time_seconds"
        ],
        time_group_edges[1:-1],
    ),
    0,
    number_of_groups - 1,
)

unique_validation_groups = np.unique(
    validation_groups
)

if len(unique_validation_groups) >= 2:
    group_cv = GroupKFold(
        n_splits=len(
            unique_validation_groups
        )
    )

    cross_validated_predictions = (
        cross_val_predict(
            context_classifier,
            X_labeled,
            y_labeled,
            groups=validation_groups,
            cv=group_cv,
            method="predict",
            n_jobs=1,
        )
    )

    print(
        "\nTime-blocked validation results:\n"
    )

    print(
        classification_report(
            y_labeled,
            cross_validated_predictions,
            zero_division=0,
        )
    )

    validation_balanced_accuracy = (
        balanced_accuracy_score(
            y_labeled,
            cross_validated_predictions,
        )
    )

    print(
        "Balanced validation accuracy:",
        round(
            validation_balanced_accuracy,
            3,
        ),
    )

    ConfusionMatrixDisplay.from_predictions(
        y_labeled,
        cross_validated_predictions,
        cmap="Blues",
        xticks_rotation=45,
    )

    plt.title(
        "Time-blocked validation confusion matrix"
    )

    plt.tight_layout()
    plt.show()

else:
    print(
        "Could not perform time-blocked validation. "
        "The labeled examples occupy only one time block."
    )



context_classifier.fit(
    X_labeled,
    y_labeled,
)

print(
    "\nContext-aware classifier fitted using "
    f"{len(labeled_features_df)} manually "
    "labeled examples."
)

#### 7.3A: Classify spike w context [must edit first cell!!]

In [ ]:

noise_label = "Noise"
isi_filtered_label = "PD"

high_confidence_probability = 0.80
supported_probability = 0.55

sequence_support_window_s = 0.200
minimum_same_label_neighbors = 2

minimum_preceding_spikes = 7
isi_standard_deviations = 4.0
minimum_isi_tolerance_ms = 4.5
apply_within_burst_isi_filter = False

experiment_name = "SC0005_03"
signal_name = "PD"

include_moderate_probability_spikes = False

##### classify spikes

In [ ]:
# ============================================================
# BLOCK 3
# CLASSIFY ALL CANDIDATES AND APPLY CONSISTENCY RULES
# ============================================================



X_all_candidates = candidate_features_df[
    waveform_feature_columns
    + context_feature_columns
]

all_predicted_labels = (
    context_classifier.predict(
        X_all_candidates
    )
)

all_prediction_probabilities = (
    context_classifier.predict_proba(
        X_all_candidates
    )
)

classifier_classes = (
    context_classifier.named_steps[
        "classifier"
    ].classes_
)

maximum_prediction_probability = np.max(
    all_prediction_probabilities,
    axis=1,
)

for class_position, class_name in enumerate(
    classifier_classes
):
    safe_class_name = (
        str(class_name)
        .replace(" ", "_")
        .replace("/", "_")
    )

    candidate_features_df[
        f"probability_{safe_class_name}"
    ] = all_prediction_probabilities[
        :,
        class_position
    ]

candidate_features_df[
    "predicted_label"
] = all_predicted_labels

candidate_features_df[
    "prediction_probability"
] = maximum_prediction_probability



predicted_labels = np.asarray(
    all_predicted_labels
).astype(str)

same_label_neighbor_count = np.zeros(
    len(candidate_features_df),
    dtype=int,
)

candidate_times_all = (
    candidate_features_df[
        "time_seconds"
    ].to_numpy(
        dtype=float
    )
)

time_sort_order = np.argsort(
    candidate_times_all
)

sorted_times = candidate_times_all[
    time_sort_order
]

sorted_labels = predicted_labels[
    time_sort_order
]

for sorted_position, event_time in enumerate(
    sorted_times
):
    left_position = np.searchsorted(
        sorted_times,
        event_time
        - sequence_support_window_s,
        side="left",
    )

    right_position = np.searchsorted(
        sorted_times,
        event_time
        + sequence_support_window_s,
        side="right",
    )

    nearby_labels = sorted_labels[
        left_position:right_position
    ]

    same_label_count = int(
        np.sum(
            nearby_labels
            == sorted_labels[
                sorted_position
            ]
        )
    ) - 1

    original_position = time_sort_order[
        sorted_position
    ]

    same_label_neighbor_count[
        original_position
    ] = max(
        0,
        same_label_count,
    )

candidate_features_df[
    "same_label_neighbor_count"
] = same_label_neighbor_count



predicted_as_neuron = (
    candidate_features_df[
        "predicted_label"
    ].astype(str)
    != noise_label
)

high_confidence = (
    candidate_features_df[
        "prediction_probability"
    ]
    >= high_confidence_probability
)

moderate_with_support = (
    (
        candidate_features_df[
            "prediction_probability"
        ]
        >= supported_probability
    )
    & (
        candidate_features_df[
            "same_label_neighbor_count"
        ]
        >= minimum_same_label_neighbors
    )
)

candidate_features_df[
    "accepted_as_spike"
] = (
    candidate_features_df[
        "valid_waveform"
    ]
    & predicted_as_neuron
    & (
        high_confidence
        | moderate_with_support
    )
)

candidate_features_df[
    "acceptance_reason"
] = np.select(
    [
        (
            predicted_as_neuron
            & high_confidence
        ),
        (
            predicted_as_neuron
            & moderate_with_support
        ),
        ~predicted_as_neuron,
    ],
    [
        "high_classifier_probability",
        "moderate_probability_with_sequence_support",
        "predicted_noise",
    ],
    default="insufficient_confidence",
)



if "reference_burst_onsets" not in globals():
    raise NameError(
        "reference_burst_onsets was not found. "
        "Run Block 1 before this block."
    )

if "reference_burst_offsets" not in globals():
    raise NameError(
        "reference_burst_offsets was not found. "
        "Run Block 1 before this block."
    )

reference_burst_onsets_array = np.asarray(
    reference_burst_onsets,
    dtype=float,
)

reference_burst_offsets_array = np.asarray(
    reference_burst_offsets,
    dtype=float,
)

candidate_burst_id = np.full(
    len(candidate_features_df),
    -1,
    dtype=int,
)

if (
    len(reference_burst_onsets_array) > 0
    and len(reference_burst_offsets_array) > 0
):
    possible_burst_ids = (
        np.searchsorted(
            reference_burst_onsets_array,
            candidate_times_all,
            side="right",
        )
        - 1
    )

    valid_possible_burst = (
        (possible_burst_ids >= 0)
        & (
            possible_burst_ids
            < len(
                reference_burst_offsets_array
            )
        )
    )

    valid_positions = np.flatnonzero(
        valid_possible_burst
    )

    valid_ids = possible_burst_ids[
        valid_possible_burst
    ]

    valid_times = candidate_times_all[
        valid_possible_burst
    ]

    inside_burst = (
        valid_times
        <= reference_burst_offsets_array[
            valid_ids
        ]
    )

    inside_positions = valid_positions[
        inside_burst
    ]

    candidate_burst_id[
        inside_positions
    ] = valid_ids[
        inside_burst
    ]

candidate_features_df[
    "reference_burst_id"
] = candidate_burst_id



candidate_features_df[
    "preceding_confirmed_spikes"
] = 0

candidate_features_df[
    "sequential_isi_seconds"
] = np.nan

candidate_features_df[
    "preceding_mean_isi_seconds"
] = np.nan

candidate_features_df[
    "preceding_isi_std_seconds"
] = np.nan

candidate_features_df[
    "maximum_allowed_isi_seconds"
] = np.nan

candidate_features_df[
    "within_burst_isi_passed"
] = True




accepted_before_isi_filter = (
    candidate_features_df[
        "accepted_as_spike"
    ].astype(bool)
)

predicted_as_isi_label = (
    candidate_features_df[
        "predicted_label"
    ].astype(str)
    == isi_filtered_label
)

eligible_for_isi_filter = (
    accepted_before_isi_filter
    & predicted_as_isi_label
    & (
        candidate_features_df[
            "reference_burst_id"
        ]
        >= 0
    )
)

eligible_burst_ids = np.unique(
    candidate_features_df.loc[
        eligible_for_isi_filter,
        "reference_burst_id",
    ].to_numpy(
        dtype=int
    )
)

number_rejected_by_isi = 0

for burst_id in eligible_burst_ids:
    burst_rows = (
        candidate_features_df.index[
            eligible_for_isi_filter
            & (
                candidate_features_df[
                    "reference_burst_id"
                ]
                == burst_id
            )
        ]
    )

    burst_rows = (
        candidate_features_df.loc[
            burst_rows
        ]
        .sort_values(
            "time_seconds"
        )
        .index
        .to_numpy()
    )

    confirmed_times = []

    for row_index in burst_rows:
        current_time = float(
            candidate_features_df.at[
                row_index,
                "time_seconds",
            ]
        )

        preceding_spike_count = len(
            confirmed_times
        )

        candidate_features_df.at[
            row_index,
            "preceding_confirmed_spikes",
        ] = preceding_spike_count

        # The first three accepted spikes seed the estimated
        # within-burst firing interval.
        if (
            preceding_spike_count
            < minimum_preceding_spikes
        ):
            confirmed_times.append(
                current_time
            )

            continue

        preceding_times_array = np.asarray(
            confirmed_times,
            dtype=float,
        )

        preceding_isis = np.diff(
            preceding_times_array
        )

        preceding_isis = preceding_isis[
            np.isfinite(
                preceding_isis
            )
            & (
                preceding_isis > 0
            )
        ]

        if len(preceding_isis) == 0:
            confirmed_times.append(
                current_time
            )

            continue

        mean_preceding_isi = float(
            np.mean(
                preceding_isis
            )
        )

        if len(preceding_isis) >= 2:
            preceding_isi_std = float(
                np.std(
                    preceding_isis,
                    ddof=1,
                )
            )
        else:
            preceding_isi_std = 0.0

        minimum_tolerance_seconds = (
            minimum_isi_tolerance_ms
            / 1000.0
        )

        effective_isi_std = max(
            preceding_isi_std,
            minimum_tolerance_seconds,
        )

        maximum_allowed_isi = (
            mean_preceding_isi
            + isi_standard_deviations
            * effective_isi_std
        )

        current_isi = (
            current_time
            - confirmed_times[-1]
        )

        isi_passed = (
            current_isi
            <= maximum_allowed_isi
        )

        candidate_features_df.at[
            row_index,
            "sequential_isi_seconds",
        ] = current_isi

        candidate_features_df.at[
            row_index,
            "preceding_mean_isi_seconds",
        ] = mean_preceding_isi

        candidate_features_df.at[
            row_index,
            "preceding_isi_std_seconds",
        ] = preceding_isi_std

        candidate_features_df.at[
            row_index,
            "maximum_allowed_isi_seconds",
        ] = maximum_allowed_isi

        candidate_features_df.at[
            row_index,
            "within_burst_isi_passed",
        ] = isi_passed

        if isi_passed:
            # This event becomes a confirmed preceding spike
            # for evaluation of the following candidate.
            confirmed_times.append(
                current_time
            )

        else:
            number_rejected_by_isi += 1

            if apply_within_burst_isi_filter:
                candidate_features_df.at[
                    row_index,
                    "accepted_as_spike",
                ] = False

                candidate_features_df.at[
                    row_index,
                    "acceptance_reason",
                ] = (
                    "rejected_by_within_"
                    "burst_maximum_isi"
                )



accepted_before_count = int(
    np.sum(
        accepted_before_isi_filter
        & predicted_as_isi_label
    )
)

accepted_after_count = int(
    np.sum(
        candidate_features_df[
            "accepted_as_spike"
        ].astype(bool)
        & predicted_as_isi_label
    )
)

print(
    f"{isi_filtered_label} accepted before "
    f"within-burst ISI filtering: "
    f"{accepted_before_count:,}"
)

print(
    f"{isi_filtered_label} rejected by the "
    f"within-burst ISI rule: "
    f"{number_rejected_by_isi:,}"
)

print(
    f"{isi_filtered_label} remaining after "
    f"within-burst ISI filtering: "
    f"{accepted_after_count:,}"
)




final_spikes_df = (
    candidate_features_df[
        candidate_features_df[
            "accepted_as_spike"
        ]
    ]
    .copy()
    .sort_values(
        "time_seconds"
    )
    .reset_index(drop=True)
)

final_spikes_df[
    "manual_label"
] = final_spikes_df[
    "predicted_label"
]

if "spike_number" in final_spikes_df.columns:
    final_spikes_df = (
        final_spikes_df.drop(
            columns="spike_number"
        )
    )

final_spikes_df.insert(
    0,
    "spike_number",
    np.arange(
        1,
        len(final_spikes_df) + 1,
    ),
)

print(
    f"\nPermissive candidates: "
    f"{len(candidate_features_df):,}"
)

print(
    f"Final accepted spikes: "
    f"{len(final_spikes_df):,}"
)

print(
    "\nFinal spikes by predicted label:"
)

display(
    final_spikes_df[
        "predicted_label"
    ]
    .value_counts()
    .rename(
        "accepted_spikes"
    )
)

display(
    final_spikes_df[
        [
            "spike_number",
            "time_seconds",
            "sample_index",
            "predicted_label",
            "prediction_probability",
            "reference_burst_id",
            "preceding_confirmed_spikes",
            "sequential_isi_seconds",
            "maximum_allowed_isi_seconds",
            "within_burst_isi_passed",
            "acceptance_reason",
        ]
    ].head(20)
)



plot_label = isi_filtered_label

plot_start_seconds = 2.0
plot_duration_seconds = 30.0


show_rejected_lp_candidates = True


acceptance_reason_colors = {
    "high_classifier_probability": "#0072B2",
    "moderate_probability_with_sequence_support": "#E69F00",
    "rejected_by_within_burst_maximum_isi": "#D55E00",
    "insufficient_confidence": "#999999",
    "predicted_noise": "#666666",
}

default_reason_color = "#CC79A7"

plot_stop_seconds = (
    plot_start_seconds
    + plot_duration_seconds
)

target_plot_mask = (
    (candidate_time >= plot_start_seconds)
    & (candidate_time <= plot_stop_seconds)
)

if not np.any(target_plot_mask):
    raise ValueError(
        "The selected diagnostic plot window contains "
        "no target-channel samples. Change "
        "plot_start_seconds."
    )



predicted_label_series = (
    candidate_features_df[
        "predicted_label"
    ].astype(str)
)

lp_candidate_mask = (
    (predicted_label_series == plot_label)
    & (
        candidate_features_df[
            "time_seconds"
        ]
        >= plot_start_seconds
    )
    & (
        candidate_features_df[
            "time_seconds"
        ]
        <= plot_stop_seconds
    )
)

lp_plot_df = (
    candidate_features_df[
        lp_candidate_mask
    ]
    .copy()
    .sort_values(
        "time_seconds"
    )
)

accepted_lp_plot_df = lp_plot_df[
    lp_plot_df[
        "accepted_as_spike"
    ].astype(bool)
].copy()

rejected_lp_plot_df = lp_plot_df[
    ~lp_plot_df[
        "accepted_as_spike"
    ].astype(bool)
].copy()




show_reference_plot = (
    "reference_time" in globals()
    and "reference_signal" in globals()
)

if show_reference_plot:
    figure, axes = plt.subplots(
        2,
        1,
        figsize=(18, 9),
        sharex=True,
        gridspec_kw={
            "height_ratios": [2, 1],
        },
    )

    target_axis = axes[0]
    reference_axis = axes[1]

else:
    figure, target_axis = plt.subplots(
        1,
        1,
        figsize=(18, 6),
    )

    reference_axis = None




target_axis.plot(
    candidate_time[
        target_plot_mask
    ],
    candidate_signal[
        target_plot_mask
    ],
    color="#B5B5B5",
    linewidth=0.7,
    label="Filtered signal",
    zorder=1,
)



accepted_reasons = (
    accepted_lp_plot_df[
        "acceptance_reason"
    ]
    .fillna(
        "unknown"
    )
    .astype(str)
    .unique()
)

for acceptance_reason in accepted_reasons:
    reason_df = accepted_lp_plot_df[
        accepted_lp_plot_df[
            "acceptance_reason"
        ]
        .fillna("unknown")
        .astype(str)
        == acceptance_reason
    ]

    reason_indices = reason_df[
        "sample_index"
    ].to_numpy(
        dtype=int
    )

    valid_indices = (
        (reason_indices >= 0)
        & (
            reason_indices
            < len(candidate_signal)
        )
    )

    reason_df = reason_df.loc[
        valid_indices
    ]

    reason_indices = reason_indices[
        valid_indices
    ]

    reason_color = (
        acceptance_reason_colors.get(
            acceptance_reason,
            default_reason_color,
        )
    )

    readable_reason = (
        acceptance_reason
        .replace("_", " ")
    )

    target_axis.scatter(
        reason_df[
            "time_seconds"
        ],
        candidate_signal[
            reason_indices
        ],
        s=65,
        marker="o",
        facecolors="none",
        edgecolors=reason_color,
        linewidths=1.8,
        label=(
            f"Accepted {plot_label}: "
            f"{readable_reason}"
        ),
        zorder=4,
    )



if show_rejected_lp_candidates:
    rejected_reasons = (
        rejected_lp_plot_df[
            "acceptance_reason"
        ]
        .fillna(
            "unknown"
        )
        .astype(str)
        .unique()
    )

    for rejection_reason in rejected_reasons:
        reason_df = rejected_lp_plot_df[
            rejected_lp_plot_df[
                "acceptance_reason"
            ]
            .fillna("unknown")
            .astype(str)
            == rejection_reason
        ]

        reason_indices = reason_df[
            "sample_index"
        ].to_numpy(
            dtype=int
        )

        valid_indices = (
            (reason_indices >= 0)
            & (
                reason_indices
                < len(candidate_signal)
            )
        )

        reason_df = reason_df.loc[
            valid_indices
        ]

        reason_indices = reason_indices[
            valid_indices
        ]

        reason_color = (
            acceptance_reason_colors.get(
                rejection_reason,
                default_reason_color,
            )
        )

        readable_reason = (
            rejection_reason
            .replace("_", " ")
        )

        target_axis.scatter(
            reason_df[
                "time_seconds"
            ],
            candidate_signal[
                reason_indices
            ],
            s=55,
            marker="x",
            color=reason_color,
            linewidths=1.7,
            label=(
                f"Rejected {plot_label}: "
                f"{readable_reason}"
            ),
            zorder=5,
        )




target_axis.set_xlim(
    plot_start_seconds,
    plot_stop_seconds,
)

target_axis.set_ylabel(
    f"{classification_channel_name}\n"
    "filtered signal"
)

target_axis.set_title(
    f"{classification_channel_name}: "
    f"{plot_label} candidates colored by acceptance reason"
)

target_axis.grid(
    visible=True,
    alpha=0.2,
)

target_axis.legend(
    loc="upper right",
    fontsize=9,
)


if reference_axis is not None:
    reference_plot_mask = (
        (
            reference_time
            >= plot_start_seconds
        )
        & (
            reference_time
            <= plot_stop_seconds
        )
    )

    reference_axis.plot(
        reference_time[
            reference_plot_mask
        ],
        reference_signal[
            reference_plot_mask
        ],
        color="#4C9ED9",
        linewidth=0.7,
        label=(
            f"Reference: "
            f"{reference_channel_name}"
        ),
    )

    
    if (
        "reference_active" in globals()
        and len(reference_active)
        == len(reference_time)
        and np.any(reference_plot_mask)
    ):
        visible_reference_signal = (
            reference_signal[
                reference_plot_mask
            ]
        )

        finite_reference_values = (
            visible_reference_signal[
                np.isfinite(
                    visible_reference_signal
                )
            ]
        )

        if len(finite_reference_values):
            reference_low = np.percentile(
                finite_reference_values,
                0.5,
            )

            reference_high = np.percentile(
                finite_reference_values,
                99.5,
            )

            reference_axis.fill_between(
                reference_time[
                    reference_plot_mask
                ],
                reference_low,
                reference_high,
                where=reference_active[
                    reference_plot_mask
                ],
                color="#55A868",
                alpha=0.12,
                label="Detected reference burst",
            )

    reference_axis.set_ylabel(
        f"{reference_channel_name}\n"
        "reference"
    )

    reference_axis.set_xlabel(
        "Time (seconds)"
    )

    reference_axis.grid(
        visible=True,
        alpha=0.2,
    )

    reference_axis.legend(
        loc="upper right",
        fontsize=9,
    )

else:
    target_axis.set_xlabel(
        "Time (seconds)"
    )


plt.tight_layout()
plt.show()




print(
    f"Accepted {plot_label} candidates "
    f"in displayed window:"
)

display(
    accepted_lp_plot_df[
        "acceptance_reason"
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "number_of_spikes"
    )
)

if show_rejected_lp_candidates:
    print(
        f"Rejected {plot_label} predictions "
        f"in displayed window:"
    )

    display(
        rejected_lp_plot_df[
            "acceptance_reason"
        ]
        .value_counts(
            dropna=False
        )
        .rename(
            "number_of_candidates"
        )
    )

In [ ]:

source_channel = channel_name

output_folder = Path(
    "classified_spike_results"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True,
)

output_filename = (
    f"{experiment_name}_"
    f"{signal_name}_"
    f"{source_channel}_"
    f"high_confidence_spikes.csv"
)

output_path = (
    output_folder
    / output_filename
)




if include_moderate_probability_spikes:
    required_reasons = [
        "high_classifier_probability",
        "moderate_probability_with_sequence_support",
    ]
else:
    required_reasons = [
        "high_classifier_probability",
    ]

if (
    "acceptance_reason"
    not in final_spikes_df.columns
):
    raise KeyError(
        "final_spikes_df does not contain an "
        "'acceptance_reason' column. Run Block 3 first."
    )

spikes_to_save = (
    final_spikes_df.loc[
        final_spikes_df[
            "acceptance_reason"
        ]
        .astype(str)
        .isin(required_reasons)
    ]
    .copy()
    .sort_values(
        "time_seconds"
    )
    .reset_index(drop=True)
)

if "spike_number" in spikes_to_save.columns:
    spikes_to_save = (
        spikes_to_save.drop(
            columns="spike_number"
        )
    )

spikes_to_save.insert(
    0,
    "spike_number",
    range(
        1,
        len(spikes_to_save) + 1,
    ),
)

print(
    "Included acceptance reasons:",
    required_reasons,
)

print(
    "Spikes retained:",
    len(spikes_to_save),
)




for column in [
    "experiment_name",
    "signal_name",
    "source_channel",
]:
    if column in spikes_to_save.columns:
        spikes_to_save = (
            spikes_to_save.drop(
                columns=column
            )
        )

spikes_to_save.insert(
    0,
    "experiment_name",
    experiment_name,
)

spikes_to_save.insert(
    1,
    "signal_name",
    signal_name,
)

spikes_to_save.insert(
    2,
    "source_channel",
    source_channel,
)



spikes_to_save.to_csv(
    output_path,
    index=False,
)

print(
    f"Saved {len(spikes_to_save):,} "
    f"high-confidence spikes to:\n"
    f"{output_path.resolve()}"
)

display(
    spikes_to_save[
        [
            "experiment_name",
            "signal_name",
            "source_channel",
            "spike_number",
            "time_seconds",
            "predicted_label",
            "prediction_probability",
            "acceptance_reason",
        ]
    ].head()
)

#### 7.2B: Alternative pipeline for subsequent data file(s)

In [ ]:
#attatch transferred predictions to the candidate features table for downstream analysis

prediction_source_df = (
    classified_candidates_df.copy()
)

print(
    "Available transferred columns:"
)

print(
    prediction_source_df.columns.tolist()
)

required_source_columns = {
    "sample_index",
    "predicted_label",
    "classification_probability",
}

missing_source_columns = (
    required_source_columns
    - set(
        prediction_source_df.columns
    )
)

if missing_source_columns:
    raise KeyError(
        "classified_candidates_df is missing "
        f"columns: {missing_source_columns}"
    )

prediction_source_df[
    "sample_index"
] = pd.to_numeric(
    prediction_source_df[
        "sample_index"
    ],
    errors="coerce",
)

prediction_source_df[
    "classification_probability"
] = pd.to_numeric(
    prediction_source_df[
        "classification_probability"
    ],
    errors="coerce",
)

prediction_source_df = (
    prediction_source_df
    .dropna(
        subset=[
            "sample_index",
        ]
    )
    .copy()
)

prediction_source_df[
    "sample_index"
] = prediction_source_df[
    "sample_index"
].astype(int)

prediction_source_df = (
    prediction_source_df
    .drop_duplicates(
        subset="sample_index",
        keep="last",
    )
)

probability_columns = [
    column
    for column
    in prediction_source_df.columns
    if column.startswith(
        "probability_"
    )
]

transfer_columns = [
    "sample_index",
    "predicted_label",
    "classification_probability",
    *probability_columns,
]

columns_to_remove_before_merge = [
    "predicted_label",
    "predicted_label_x",
    "predicted_label_y",
    "classification_probability",
    "classification_probability_x",
    "classification_probability_y",
    "prediction_probability",
]

columns_to_remove_before_merge.extend([
    column
    for column
    in candidate_features_df.columns
    if column.startswith(
        "probability_"
    )
])

columns_to_remove_before_merge = [
    column
    for column
    in columns_to_remove_before_merge
    if column
    in candidate_features_df.columns
]

if columns_to_remove_before_merge:
    candidate_features_df = (
        candidate_features_df.drop(
            columns=columns_to_remove_before_merge
        )
    )

candidate_features_df[
    "sample_index"
] = pd.to_numeric(
    candidate_features_df[
        "sample_index"
    ],
    errors="coerce",
)

candidate_features_df = (
    candidate_features_df
    .dropna(
        subset=[
            "sample_index",
        ]
    )
    .copy()
)

candidate_features_df[
    "sample_index"
] = candidate_features_df[
    "sample_index"
].astype(int)

candidate_features_df = (
    candidate_features_df.merge(
        prediction_source_df[
            transfer_columns
        ],
        on="sample_index",
        how="left",
        validate="one_to_one",
    )
)

matched_predictions = (
    candidate_features_df[
        "predicted_label"
    ].notna()
)

print(
    "Rhythmic feature candidates:",
    len(candidate_features_df),
)

print(
    "Transferred candidate rows:",
    len(prediction_source_df),
)

print(
    "Matched predictions:",
    int(
        matched_predictions.sum()
    ),
)

print(
    "Unmatched candidates:",
    int(
        (~matched_predictions).sum()
    ),
)

print(
    "\nMatched predicted labels:"
)

print(
    candidate_features_df[
        "predicted_label"
    ].value_counts(
        dropna=False
    )
)

In [ ]:
#apply consistency rules to transffered predictions 

plot_label = isi_filtered_label
plot_start_seconds = float(candidate_time[0])
plot_duration_seconds = 30.0

show_rejected_candidates = True

acceptance_reason_colors = {
    "high_classifier_probability": "#0072B2",
    "moderate_probability_with_sequence_support": "#E69F00",
    "rejected_by_within_burst_maximum_isi": "#D55E00",
    "insufficient_confidence": "#999999",
    "predicted_noise": "#666666",
    "invalid_waveform": "#CC79A7",
    "missing_transferred_prediction": "#000000",
}

default_reason_color = "#CC79A7"




required_transfer_columns = {
    "predicted_label",
    "classification_probability",
    "sample_index",
    "time_seconds",
    "valid_waveform",
}

missing_transfer_columns = (
    required_transfer_columns
    - set(
        candidate_features_df.columns
    )
)

if missing_transfer_columns:
    raise KeyError(
        "candidate_features_df is missing "
        "transferred-classifier columns: "
        f"{missing_transfer_columns}"
    )




candidate_features_df[
    "predicted_label"
] = (
    candidate_features_df[
        "predicted_label"
    ]
    .fillna("Unmatched")
    .astype(str)
    .str.strip()
)

candidate_features_df[
    "prediction_probability"
] = pd.to_numeric(
    candidate_features_df[
        "classification_probability"
    ],
    errors="coerce",
)

all_predicted_labels = (
    candidate_features_df[
        "predicted_label"
    ].to_numpy(
        dtype=str
    )
)

missing_transferred_prediction = (
    candidate_features_df[
        "predicted_label"
    ].isin(
        [
            "Unmatched",
            "nan",
            "None",
        ]
    )
    | candidate_features_df[
        "prediction_probability"
    ].isna()
)

candidate_features_df.loc[
    missing_transferred_prediction,
    "predicted_label",
] = "Unmatched"

all_predicted_labels = (
    candidate_features_df[
        "predicted_label"
    ].to_numpy(
        dtype=str
    )
)

print(
    "Transferred predicted labels:"
)

print(
    candidate_features_df[
        "predicted_label"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nTransferred probability summary:"
)

print(
    candidate_features_df[
        "prediction_probability"
    ].describe()
)

print(
    "\nCandidates without transferred predictions:",
    int(
        missing_transferred_prediction.sum()
    ),
)



same_label_neighbor_count = np.zeros(
    len(candidate_features_df),
    dtype=int,
)

candidate_times_all = (
    candidate_features_df[
        "time_seconds"
    ].to_numpy(
        dtype=float
    )
)

time_sort_order = np.argsort(
    candidate_times_all
)

sorted_times = candidate_times_all[
    time_sort_order
]

sorted_labels = all_predicted_labels[
    time_sort_order
]

for sorted_position, event_time in enumerate(
    sorted_times
):
    left_position = np.searchsorted(
        sorted_times,
        (
            event_time
            - sequence_support_window_s
        ),
        side="left",
    )

    right_position = np.searchsorted(
        sorted_times,
        (
            event_time
            + sequence_support_window_s
        ),
        side="right",
    )

    nearby_labels = sorted_labels[
        left_position:right_position
    ]

    same_label_count = int(
        np.sum(
            nearby_labels
            == sorted_labels[
                sorted_position
            ]
        )
    ) - 1

    original_position = time_sort_order[
        sorted_position
    ]

    same_label_neighbor_count[
        original_position
    ] = max(
        0,
        same_label_count,
    )

candidate_features_df[
    "same_label_neighbor_count"
] = same_label_neighbor_count



valid_waveform_mask = (
    candidate_features_df[
        "valid_waveform"
    ]
    .fillna(False)
    .astype(bool)
)

target_label_mask = (
    candidate_features_df[
        "predicted_label"
    ]
    == isi_filtered_label
)

noise_label_mask = (
    candidate_features_df[
        "predicted_label"
    ]
    == noise_label
)

high_confidence_mask = (
    target_label_mask
    & valid_waveform_mask
    & (
        candidate_features_df[
            "prediction_probability"
        ]
        >= high_confidence_probability
    )
)

moderate_confidence_mask = (
    target_label_mask
    & valid_waveform_mask
    & ~high_confidence_mask
    & (
        candidate_features_df[
            "prediction_probability"
        ]
        >= supported_probability
    )
    & (
        candidate_features_df[
            "same_label_neighbor_count"
        ]
        >= minimum_same_label_neighbors
    )
)

candidate_features_df[
    "accepted_as_spike"
] = (
    high_confidence_mask
    | moderate_confidence_mask
)

candidate_features_df[
    "acceptance_reason"
] = "insufficient_confidence"

candidate_features_df.loc[
    noise_label_mask,
    "acceptance_reason",
] = "predicted_noise"

candidate_features_df.loc[
    moderate_confidence_mask,
    "acceptance_reason",
] = (
    "moderate_probability_with_sequence_support"
)

candidate_features_df.loc[
    high_confidence_mask,
    "acceptance_reason",
] = "high_classifier_probability"

candidate_features_df.loc[
    ~valid_waveform_mask,
    "acceptance_reason",
] = "invalid_waveform"

candidate_features_df.loc[
    missing_transferred_prediction,
    "acceptance_reason",
] = "missing_transferred_prediction"

print(
    "\nTarget label:",
    isi_filtered_label,
)

print(
    "Noise label:",
    noise_label,
)

print(
    "High-confidence target candidates:",
    int(
        high_confidence_mask.sum()
    ),
)

print(
    "Moderate supported target candidates:",
    int(
        moderate_confidence_mask.sum()
    ),
)

print(
    "Accepted before ISI filtering:",
    int(
        candidate_features_df[
            "accepted_as_spike"
        ].sum()
    ),
)

print(
    "\nAcceptance reasons before ISI filtering:"
)

print(
    candidate_features_df[
        "acceptance_reason"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nAccepted labels before ISI filtering:"
)

print(
    candidate_features_df.loc[
        candidate_features_df[
            "accepted_as_spike"
        ],
        "predicted_label",
    ].value_counts(
        dropna=False
    )
)



if "reference_burst_onsets" not in globals():
    raise NameError(
        "reference_burst_onsets was not found. "
        "Run waveform/temporal/rhythmic Block 1 first."
    )

if "reference_burst_offsets" not in globals():
    raise NameError(
        "reference_burst_offsets was not found. "
        "Run waveform/temporal/rhythmic Block 1 first."
    )

reference_burst_onsets_array = np.asarray(
    reference_burst_onsets,
    dtype=float,
)

reference_burst_offsets_array = np.asarray(
    reference_burst_offsets,
    dtype=float,
)

if (
    len(reference_burst_onsets_array)
    != len(reference_burst_offsets_array)
):
    raise ValueError(
        "Reference burst onset and offset arrays "
        "have different lengths."
    )

candidate_burst_id = np.full(
    len(candidate_features_df),
    -1,
    dtype=int,
)

if (
    len(reference_burst_onsets_array) > 0
    and len(reference_burst_offsets_array) > 0
):
    possible_burst_ids = (
        np.searchsorted(
            reference_burst_onsets_array,
            candidate_times_all,
            side="right",
        )
        - 1
    )

    valid_possible_burst = (
        (possible_burst_ids >= 0)
        & (
            possible_burst_ids
            < len(
                reference_burst_offsets_array
            )
        )
    )

    if (
        "reference_reliable"
        in candidate_features_df.columns
    ):
        valid_possible_burst = (
            valid_possible_burst
            & (
                candidate_features_df[
                    "reference_reliable"
                ]
                .fillna(0)
                .astype(bool)
                .to_numpy()
            )
        )

    valid_positions = np.flatnonzero(
        valid_possible_burst
    )

    valid_ids = possible_burst_ids[
        valid_possible_burst
    ]

    valid_times = candidate_times_all[
        valid_possible_burst
    ]

    inside_burst = (
        valid_times
        <= reference_burst_offsets_array[
            valid_ids
        ]
    )

    inside_positions = valid_positions[
        inside_burst
    ]

    candidate_burst_id[
        inside_positions
    ] = valid_ids[
        inside_burst
    ]

candidate_features_df[
    "reference_burst_id"
] = candidate_burst_id

print(
    "\nCandidates inside reliable reference bursts:",
    int(
        np.sum(
            candidate_burst_id >= 0
        )
    ),
)

print(
    "Candidates outside reliable reference bursts:",
    int(
        np.sum(
            candidate_burst_id < 0
        )
    ),
)



candidate_features_df[
    "preceding_confirmed_spikes"
] = 0

candidate_features_df[
    "sequential_isi_seconds"
] = np.nan

candidate_features_df[
    "preceding_mean_isi_seconds"
] = np.nan

candidate_features_df[
    "preceding_isi_std_seconds"
] = np.nan

candidate_features_df[
    "maximum_allowed_isi_seconds"
] = np.nan

candidate_features_df[
    "within_burst_isi_passed"
] = True


accepted_before_isi_filter = (
    candidate_features_df[
        "accepted_as_spike"
    ]
    .fillna(False)
    .astype(bool)
    .copy()
)

predicted_as_isi_label = (
    candidate_features_df[
        "predicted_label"
    ]
    == isi_filtered_label
)

eligible_for_isi_filter = (
    accepted_before_isi_filter
    & predicted_as_isi_label
    & (
        candidate_features_df[
            "reference_burst_id"
        ]
        >= 0
    )
)

eligible_burst_ids = np.unique(
    candidate_features_df.loc[
        eligible_for_isi_filter,
        "reference_burst_id",
    ].to_numpy(
        dtype=int
    )
)

number_rejected_by_isi = 0

for burst_id in eligible_burst_ids:
    burst_rows = (
        candidate_features_df.index[
            eligible_for_isi_filter
            & (
                candidate_features_df[
                    "reference_burst_id"
                ]
                == burst_id
            )
        ]
    )

    burst_rows = (
        candidate_features_df.loc[
            burst_rows
        ]
        .sort_values(
            "time_seconds"
        )
        .index
        .to_numpy()
    )

    confirmed_times = []

    for row_index in burst_rows:
        current_time = float(
            candidate_features_df.at[
                row_index,
                "time_seconds",
            ]
        )

        preceding_spike_count = len(
            confirmed_times
        )

        candidate_features_df.at[
            row_index,
            "preceding_confirmed_spikes",
        ] = preceding_spike_count

        if (
            preceding_spike_count
            < minimum_preceding_spikes
        ):
            confirmed_times.append(
                current_time
            )

            continue

        preceding_times_array = np.asarray(
            confirmed_times,
            dtype=float,
        )

        preceding_isis = np.diff(
            preceding_times_array
        )

        preceding_isis = preceding_isis[
            np.isfinite(
                preceding_isis
            )
            & (
                preceding_isis > 0
            )
        ]

        if len(preceding_isis) == 0:
            confirmed_times.append(
                current_time
            )

            continue

        mean_preceding_isi = float(
            np.mean(
                preceding_isis
            )
        )

        if len(preceding_isis) >= 2:
            preceding_isi_std = float(
                np.std(
                    preceding_isis,
                    ddof=1,
                )
            )
        else:
            preceding_isi_std = 0.0

        minimum_tolerance_seconds = (
            minimum_isi_tolerance_ms
            / 1000.0
        )

        effective_isi_std = max(
            preceding_isi_std,
            minimum_tolerance_seconds,
        )

        maximum_allowed_isi = (
            mean_preceding_isi
            + isi_standard_deviations
            * effective_isi_std
        )

        current_isi = (
            current_time
            - confirmed_times[-1]
        )

        isi_passed = (
            current_isi
            <= maximum_allowed_isi
        )

        candidate_features_df.at[
            row_index,
            "sequential_isi_seconds",
        ] = current_isi

        candidate_features_df.at[
            row_index,
            "preceding_mean_isi_seconds",
        ] = mean_preceding_isi

        candidate_features_df.at[
            row_index,
            "preceding_isi_std_seconds",
        ] = preceding_isi_std

        candidate_features_df.at[
            row_index,
            "maximum_allowed_isi_seconds",
        ] = maximum_allowed_isi

        candidate_features_df.at[
            row_index,
            "within_burst_isi_passed",
        ] = isi_passed

        if isi_passed:
            confirmed_times.append(
                current_time
            )

        else:
            number_rejected_by_isi += 1

            if apply_within_burst_isi_filter:
                candidate_features_df.at[
                    row_index,
                    "accepted_as_spike",
                ] = False

                candidate_features_df.at[
                    row_index,
                    "acceptance_reason",
                ] = (
                    "rejected_by_within_burst_"
                    "maximum_isi"
                )




accepted_before_count = int(
    np.sum(
        accepted_before_isi_filter
        & predicted_as_isi_label
    )
)

accepted_after_count = int(
    np.sum(
        candidate_features_df[
            "accepted_as_spike"
        ]
        .fillna(False)
        .astype(bool)
        & predicted_as_isi_label
    )
)

print(
    f"\n{isi_filtered_label} accepted before "
    f"within-burst ISI filtering: "
    f"{accepted_before_count:,}"
)

print(
    f"{isi_filtered_label} rejected by the "
    f"within-burst ISI rule: "
    f"{number_rejected_by_isi:,}"
)

print(
    f"{isi_filtered_label} remaining after "
    f"within-burst ISI filtering: "
    f"{accepted_after_count:,}"
)



accepted_final_mask = (
    candidate_features_df[
        "accepted_as_spike"
    ]
    .fillna(False)
    .astype(bool)
)

final_spikes_df = (
    candidate_features_df.loc[
        accepted_final_mask
    ]
    .copy()
    .sort_values(
        "time_seconds"
    )
    .reset_index(drop=True)
)

final_spikes_df[
    "manual_label"
] = final_spikes_df[
    "predicted_label"
]

if "spike_number" in final_spikes_df.columns:
    final_spikes_df = (
        final_spikes_df.drop(
            columns="spike_number"
        )
    )

final_spikes_df.insert(
    0,
    "spike_number",
    np.arange(
        1,
        len(final_spikes_df) + 1,
    ),
)

print(
    f"\nPermissive candidates: "
    f"{len(candidate_features_df):,}"
)

print(
    f"Final accepted spikes: "
    f"{len(final_spikes_df):,}"
)

print(
    "\nFinal spikes by predicted label:"
)

display(
    final_spikes_df[
        "predicted_label"
    ]
    .value_counts()
    .rename(
        "accepted_spikes"
    )
)

display_columns = [
    "spike_number",
    "time_seconds",
    "sample_index",
    "predicted_label",
    "prediction_probability",
    "reference_burst_id",
    "preceding_confirmed_spikes",
    "sequential_isi_seconds",
    "maximum_allowed_isi_seconds",
    "within_burst_isi_passed",
    "acceptance_reason",
]

display(
    final_spikes_df[
        display_columns
    ].head(20)
)



plot_stop_seconds = (
    plot_start_seconds
    + plot_duration_seconds
)

target_plot_mask = (
    (candidate_time >= plot_start_seconds)
    & (candidate_time <= plot_stop_seconds)
)

if not np.any(target_plot_mask):
    raise ValueError(
        "The selected diagnostic plot window contains "
        "no target-channel samples. Change "
        "plot_start_seconds."
    )

predicted_label_series = (
    candidate_features_df[
        "predicted_label"
    ].astype(str)
)

plot_candidate_mask = (
    (predicted_label_series == plot_label)
    & (
        candidate_features_df[
            "time_seconds"
        ]
        >= plot_start_seconds
    )
    & (
        candidate_features_df[
            "time_seconds"
        ]
        <= plot_stop_seconds
    )
)

candidate_plot_df = (
    candidate_features_df.loc[
        plot_candidate_mask
    ]
    .copy()
    .sort_values(
        "time_seconds"
    )
)

accepted_plot_df = (
    candidate_plot_df.loc[
        candidate_plot_df[
            "accepted_as_spike"
        ]
        .fillna(False)
        .astype(bool)
    ]
    .copy()
)

rejected_plot_df = (
    candidate_plot_df.loc[
        ~candidate_plot_df[
            "accepted_as_spike"
        ]
        .fillna(False)
        .astype(bool)
    ]
    .copy()
)

show_reference_plot = (
    "reference_time" in globals()
    and "reference_signal" in globals()
)

if show_reference_plot:
    figure, axes = plt.subplots(
        2,
        1,
        figsize=(18, 9),
        sharex=True,
        gridspec_kw={
            "height_ratios": [
                2,
                1,
            ],
        },
    )

    target_axis = axes[0]
    reference_axis = axes[1]

else:
    figure, target_axis = plt.subplots(
        1,
        1,
        figsize=(18, 6),
    )

    reference_axis = None

target_axis.plot(
    candidate_time[
        target_plot_mask
    ],
    candidate_signal[
        target_plot_mask
    ],
    color="#B5B5B5",
    linewidth=0.7,
    label="Filtered signal",
    zorder=1,
)

accepted_reasons = (
    accepted_plot_df[
        "acceptance_reason"
    ]
    .fillna("unknown")
    .astype(str)
    .unique()
)

for acceptance_reason in accepted_reasons:
    reason_df = accepted_plot_df.loc[
        accepted_plot_df[
            "acceptance_reason"
        ]
        .fillna("unknown")
        .astype(str)
        == acceptance_reason
    ]

    reason_indices = reason_df[
        "sample_index"
    ].to_numpy(
        dtype=int
    )

    valid_indices = (
        (reason_indices >= 0)
        & (
            reason_indices
            < len(candidate_signal)
        )
    )

    reason_df = reason_df.loc[
        valid_indices
    ]

    reason_indices = reason_indices[
        valid_indices
    ]

    reason_color = acceptance_reason_colors.get(
        acceptance_reason,
        default_reason_color,
    )

    readable_reason = (
        acceptance_reason.replace(
            "_",
            " ",
        )
    )

    target_axis.scatter(
        reason_df[
            "time_seconds"
        ],
        candidate_signal[
            reason_indices
        ],
        s=65,
        marker="o",
        facecolors="none",
        edgecolors=reason_color,
        linewidths=1.8,
        label=(
            f"Accepted {plot_label}: "
            f"{readable_reason}"
        ),
        zorder=4,
    )

if show_rejected_candidates:
    rejected_reasons = (
        rejected_plot_df[
            "acceptance_reason"
        ]
        .fillna("unknown")
        .astype(str)
        .unique()
    )

    for rejection_reason in rejected_reasons:
        reason_df = rejected_plot_df.loc[
            rejected_plot_df[
                "acceptance_reason"
            ]
            .fillna("unknown")
            .astype(str)
            == rejection_reason
        ]

        reason_indices = reason_df[
            "sample_index"
        ].to_numpy(
            dtype=int
        )

        valid_indices = (
            (reason_indices >= 0)
            & (
                reason_indices
                < len(candidate_signal)
            )
        )

        reason_df = reason_df.loc[
            valid_indices
        ]

        reason_indices = reason_indices[
            valid_indices
        ]

        reason_color = acceptance_reason_colors.get(
            rejection_reason,
            default_reason_color,
        )

        readable_reason = (
            rejection_reason.replace(
                "_",
                " ",
            )
        )

        target_axis.scatter(
            reason_df[
                "time_seconds"
            ],
            candidate_signal[
                reason_indices
            ],
            s=55,
            marker="x",
            color=reason_color,
            linewidths=1.7,
            label=(
                f"Rejected {plot_label}: "
                f"{readable_reason}"
            ),
            zorder=5,
        )

target_axis.set_xlim(
    plot_start_seconds,
    plot_stop_seconds,
)

target_axis.set_ylabel(
    f"{classification_channel_name}\n"
    "filtered signal"
)

target_axis.set_title(
    f"{classification_channel_name}: "
    f"{plot_label} candidates colored by "
    "acceptance reason"
)

target_axis.grid(
    visible=True,
    alpha=0.2,
)

target_axis.legend(
    loc="upper right",
    fontsize=9,
)

if reference_axis is not None:
    reference_plot_mask = (
        (
            reference_time
            >= plot_start_seconds
        )
        & (
            reference_time
            <= plot_stop_seconds
        )
    )

    reference_axis.plot(
        reference_time[
            reference_plot_mask
        ],
        reference_signal[
            reference_plot_mask
        ],
        color="#4C9ED9",
        linewidth=0.7,
        label=(
            f"Reference: "
            f"{reference_channel_name}"
        ),
    )

    if (
        "reference_active" in globals()
        and len(reference_active)
        == len(reference_time)
        and np.any(reference_plot_mask)
    ):
        visible_reference_signal = (
            reference_signal[
                reference_plot_mask
            ]
        )

        finite_reference_values = (
            visible_reference_signal[
                np.isfinite(
                    visible_reference_signal
                )
            ]
        )

        if len(finite_reference_values):
            reference_low = np.percentile(
                finite_reference_values,
                0.5,
            )

            reference_high = np.percentile(
                finite_reference_values,
                99.5,
            )

            reference_axis.fill_between(
                reference_time[
                    reference_plot_mask
                ],
                reference_low,
                reference_high,
                where=reference_active[
                    reference_plot_mask
                ],
                color="#55A868",
                alpha=0.12,
                label="Detected reference burst",
            )

    reference_axis.set_ylabel(
        f"{reference_channel_name}\n"
        "reference"
    )

    reference_axis.set_xlabel(
        "Time (seconds)"
    )

    reference_axis.grid(
        visible=True,
        alpha=0.2,
    )

    reference_axis.legend(
        loc="upper right",
        fontsize=9,
    )

else:
    target_axis.set_xlabel(
        "Time (seconds)"
    )

plt.tight_layout()
plt.show()

print(
    f"Accepted {plot_label} candidates "
    f"in displayed window:"
)

display(
    accepted_plot_df[
        "acceptance_reason"
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "number_of_spikes"
    )
)

if show_rejected_candidates:
    print(
        f"Rejected {plot_label} predictions "
        f"in displayed window:"
    )

    display(
        rejected_plot_df[
            "acceptance_reason"
        ]
        .value_counts(
            dropna=False
        )
        .rename(
            "number_of_candidates"
        )
    )

In [ ]:

source_channel = channel_name

output_folder = Path(
    "classified_spike_results"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True,
)

output_filename = (
    f"{experiment_name}_"
    f"{signal_name}_"
    f"{source_channel}_"
    f"high_confidence_spikes.csv"
)

output_path = (
    output_folder
    / output_filename
)




if include_moderate_probability_spikes:
    required_reasons = [
        "high_classifier_probability",
        "moderate_probability_with_sequence_support",
    ]
else:
    required_reasons = [
        "high_classifier_probability",
    ]

if (
    "acceptance_reason"
    not in final_spikes_df.columns
):
    raise KeyError(
        "final_spikes_df does not contain an "
        "'acceptance_reason' column. Run Block 3 first."
    )

spikes_to_save = (
    final_spikes_df.loc[
        final_spikes_df[
            "acceptance_reason"
        ]
        .astype(str)
        .isin(required_reasons)
    ]
    .copy()
    .sort_values(
        "time_seconds"
    )
    .reset_index(drop=True)
)

if "spike_number" in spikes_to_save.columns:
    spikes_to_save = (
        spikes_to_save.drop(
            columns="spike_number"
        )
    )

spikes_to_save.insert(
    0,
    "spike_number",
    range(
        1,
        len(spikes_to_save) + 1,
    ),
)

print(
    "Included acceptance reasons:",
    required_reasons,
)

print(
    "Spikes retained:",
    len(spikes_to_save),
)

# ============================================================
# ADD EXPERIMENT INFORMATION
# ============================================================

# Avoid duplicate columns if this block is rerun.
for column in [
    "experiment_name",
    "signal_name",
    "source_channel",
]:
    if column in spikes_to_save.columns:
        spikes_to_save = (
            spikes_to_save.drop(
                columns=column
            )
        )

spikes_to_save.insert(
    0,
    "experiment_name",
    experiment_name,
)

spikes_to_save.insert(
    1,
    "signal_name",
    signal_name,
)

spikes_to_save.insert(
    2,
    "source_channel",
    source_channel,
)




spikes_to_save.to_csv(
    output_path,
    index=False,
)

print(
    f"Saved {len(spikes_to_save):,} "
    f"high-confidence spikes to:\n"
    f"{output_path.resolve()}"
)

display(
    spikes_to_save[
        [
            "experiment_name",
            "signal_name",
            "source_channel",
            "spike_number",
            "time_seconds",
            "predicted_label",
            "prediction_probability",
            "acceptance_reason",
        ]
    ].head()
)

### Step 8: Check spike times in gui

##### settings for where to pull sorted spikes 

In [ ]:
# Folder containing labeled sorted-spike CSV files
sorted_csv_directory = Path(
   '/Volumes/marder-lab/helmore/marder_py/classified_spike_results/SC0005'
)

#sorted_csv_pattern = "*_sorted_spikes.csv"
sorted_csv_pattern = "*02*.csv"

##### gui

In [ ]:

# Labels excluded from the viewer
excluded_labels = {
    "noise",
    "unlabeled",
    "uncertain",
}

# Initial view
initial_window_seconds = 30.0
initial_signal_mode = "Filtered"

# Playback
initial_play_step_seconds = 0.5
play_interval_ms = 150

# Rendering
use_opengl = True
trace_color = "#9a9a9a"
trace_width = 1.0
spike_marker_size = 9

# Limits the number of markers rendered for one label in one
# visible window. This affects display only, not saved data.
maximum_visible_markers_per_label = 20_000

# Percentiles used for automatic Y scaling
y_percentile_low = 0.5
y_percentile_high = 99.5

# Debounce interval prevents redundant updates while dragging
update_debounce_ms = 16


# ============================================================
# CONFIGURE PYQTGRAPH
# ============================================================

pg.setConfigOptions(
    antialias=False,
    useOpenGL=use_opengl,
)

if not sorted_csv_directory.exists():
    raise FileNotFoundError(
        f"CSV directory does not exist: "
        f"{sorted_csv_directory}"
    )

if "processed_channels" not in globals():
    raise NameError(
        "processed_channels is unavailable. Run the SMR "
        "loading, masking, and filtering code first."
    )

csv_files = sorted(
    sorted_csv_directory.glob(
        sorted_csv_pattern
    )
)

if not csv_files:
    raise FileNotFoundError(
        f"No files matching {sorted_csv_pattern!r} "
        f"were found in {sorted_csv_directory}."
    )

available_channel_names = list(
    processed_channels.keys()
)


# ============================================================
# CHANNEL MATCHING
# ============================================================

def match_channel_name(
    candidate,
    available_names,
):
    candidate = str(
        candidate
    ).strip().lower()

    exact = [
        name
        for name in available_names
        if str(name).strip().lower()
        == candidate
    ]

    if len(exact) == 1:
        return exact[0]

    partial = [
        name
        for name in available_names
        if (
            candidate
            in str(name).lower()
            or str(name).lower()
            in candidate
        )
    ]

    if len(partial) == 1:
        return partial[0]

    return None


def match_filename_to_channel(
    csv_file,
    available_names,
):
    filename = csv_file.stem.lower()

    matches = [
        name
        for name in available_names
        if str(name).lower()
        in filename
    ]

    if not matches:
        return None

    # Prefer the longest matching name
    matches.sort(
        key=lambda value: len(str(value)),
        reverse=True,
    )

    return matches[0]


# ============================================================
# LOAD SORTED-SPIKE FILES
# ============================================================

spike_tables_by_channel = {}

for csv_file in csv_files:
    csv_data = pd.read_csv(
        csv_file
    )

    # Prefer the polarity-aware aligned waveform time
    if (
        "aligned_time_seconds"
        in csv_data.columns
    ):
        time_column = (
            "aligned_time_seconds"
        )

    elif "time_seconds" in csv_data.columns:
        time_column = "time_seconds"

    else:
        print(
            f"Skipping {csv_file.name}: "
            "no spike-time column."
        )
        continue

    if "manual_label" in csv_data.columns:
        label_column = "manual_label"

    elif "cluster_id" in csv_data.columns:
        label_column = "cluster_id"

    else:
        csv_data["_viewer_default_label"] = (
            "spike"
        )

        label_column = (
            "_viewer_default_label"
        )

    channel_groups = []

    if "source_channel" in csv_data.columns:
        source_values = (
            csv_data["source_channel"]
            .dropna()
            .astype(str)
            .unique()
        )

        for source_value in source_values:
            matched_channel = (
                match_channel_name(
                    source_value,
                    available_channel_names,
                )
            )

            if matched_channel is None:
                print(
                    f"Could not match channel "
                    f"{source_value!r} from "
                    f"{csv_file.name}."
                )
                continue

            group_mask = (
                csv_data[
                    "source_channel"
                ].astype(str)
                == source_value
            )

            channel_groups.append((
                matched_channel,
                csv_data[group_mask].copy(),
            ))

    else:
        matched_channel = (
            match_filename_to_channel(
                csv_file,
                available_channel_names,
            )
        )

        if matched_channel is None:
            print(
                f"Could not match {csv_file.name} "
                "to a processed channel."
            )
            continue

        channel_groups.append((
            matched_channel,
            csv_data.copy(),
        ))

    for channel_name, table in channel_groups:
        table["_viewer_time"] = pd.to_numeric(
            table[time_column],
            errors="coerce",
        )

        table["_viewer_label"] = (
            table[label_column]
            .astype(str)
            .str.strip()
        )

        table = table.dropna(
            subset=["_viewer_time"]
        )

        excluded_mask = (
            table["_viewer_label"]
            .str.lower()
            .isin(excluded_labels)
        )

        table = table[
            ~excluded_mask
        ]

        if table.empty:
            continue

        spike_tables_by_channel.setdefault(
            channel_name,
            [],
        ).append(table)

        print(
            f"Loaded {len(table):,} spikes: "
            f"{csv_file.name} → {channel_name}"
        )


# ============================================================
# COMBINE TABLES FOR EACH CHANNEL
# ============================================================

sorted_spikes_by_channel = {}

for channel_name, tables in (
    spike_tables_by_channel.items()
):
    combined = pd.concat(
        tables,
        ignore_index=True,
    )

    combined = (
        combined
        .sort_values("_viewer_time")
        .reset_index(drop=True)
    )

    sorted_spikes_by_channel[
        channel_name
    ] = combined


channels_to_display = [
    channel_name
    for channel_name
    in available_channel_names
    if channel_name
    in sorted_spikes_by_channel
]

if not channels_to_display:
    raise ValueError(
        "No sorted-spike files matched the available "
        "processed channels."
    )

print("\nChannels displayed:")

for channel_name in channels_to_display:
    print(
        f"  {channel_name}: "
        f"{len(sorted_spikes_by_channel[channel_name]):,} "
        "spikes"
    )


# ============================================================
# LABEL COLORS
# ============================================================

all_labels = sorted({
    str(label)
    for channel_name
    in channels_to_display
    for label
    in sorted_spikes_by_channel[
        channel_name
    ]["_viewer_label"].unique()
})

default_colors = [
    "#e41a1c",
    "#377eb8",
    "#4daf4a",
    "#984ea3",
    "#ff7f00",
    "#a65628",
    "#f781bf",
    "#17becf",
    "#bcbd22",
]

label_colors = {
    label: default_colors[
        index % len(default_colors)
    ]
    for index, label in enumerate(
        all_labels
    )
}


# ============================================================
# OPTIMIZED VIEWER
# ============================================================

class SortedSpikeViewer(
    QtWidgets.QMainWindow
):
    def __init__(
        self,
        processed_channels,
        spikes_by_channel,
        channel_names,
        label_colors,
    ):
        super().__init__()

        self.processed_channels = (
            processed_channels
        )

        self.spikes_by_channel = (
            spikes_by_channel
        )

        self.channel_names = list(
            channel_names
        )

        self.label_colors = dict(
            label_colors
        )

        self.data_cache = {}
        self.plot_records = {}
        self.y_zoom_boxes = {}

        self.first_plot = None
        self.updating_controls = False

        self.play_timer = QtCore.QTimer(
            self
        )

        self.play_timer.setInterval(
            play_interval_ms
        )

        self.play_timer.timeout.connect(
            self.advance_playback
        )

        # Debounces slider and spin-box updates
        self.refresh_timer = QtCore.QTimer(
            self
        )

        self.refresh_timer.setSingleShot(
            True
        )

        self.refresh_timer.setInterval(
            update_debounce_ms
        )

        self.refresh_timer.timeout.connect(
            self.update_view
        )

        self.setWindowTitle(
            "Sorted extracellular spike viewer"
        )

        self.resize(
            1550,
            920,
        )

        self.build_data_cache()
        self.build_interface()
        self.build_plots()
        self.initialize_ranges()
        self.install_shortcuts()
        self.update_signal_mode()
        self.update_view()

    # --------------------------------------------------------
    # Cache data as NumPy arrays once
    # --------------------------------------------------------

    def build_data_cache(self):
        all_spike_times = []

        for channel_name in self.channel_names:
            result = self.processed_channels[
                channel_name
            ]

            channel_time = np.asarray(
                result["time"],
                dtype=np.float64,
            )

            filtered = np.asarray(
                result["filtered_signal"],
                dtype=np.float32,
            )

            raw = None

            if "raw_signal" in result:
                raw = np.asarray(
                    result["raw_signal"],
                    dtype=np.float32,
                )

            if len(channel_time) != len(filtered):
                raise ValueError(
                    f"{channel_name}: time and filtered "
                    "signal lengths differ."
                )

            spike_table = (
                self.spikes_by_channel[
                    channel_name
                ]
            )

            label_spikes = {}

            for label in sorted(
                spike_table[
                    "_viewer_label"
                ].unique()
            ):
                label_times = (
                    spike_table.loc[
                        spike_table[
                            "_viewer_label"
                        ]
                        == label,
                        "_viewer_time",
                    ]
                    .to_numpy(dtype=np.float64)
                )

                label_times = np.sort(
                    label_times[
                        np.isfinite(label_times)
                    ]
                )

                label_spikes[
                    str(label)
                ] = label_times

                all_spike_times.append(
                    label_times
                )

            self.data_cache[
                channel_name
            ] = {
                "time": channel_time,
                "Filtered": filtered,
                "Raw": raw,
                "spikes": label_spikes,
            }

        nonempty_arrays = [
            values
            for values in all_spike_times
            if len(values)
        ]

        if nonempty_arrays:
            self.all_spike_times = np.unique(
                np.concatenate(
                    nonempty_arrays
                )
            )

        else:
            self.all_spike_times = np.array(
                [],
                dtype=np.float64,
            )

    # --------------------------------------------------------
    # Main interface
    # --------------------------------------------------------

    def build_interface(self):
        central = QtWidgets.QWidget()
        self.setCentralWidget(central)

        outer_layout = (
            QtWidgets.QHBoxLayout(
                central
            )
        )

        left_widget = QtWidgets.QWidget()

        left_layout = (
            QtWidgets.QVBoxLayout(
                left_widget
            )
        )

        controls = QtWidgets.QHBoxLayout()

        self.play_button = (
            QtWidgets.QPushButton(
                "Play"
            )
        )

        self.play_button.setCheckable(
            True
        )

        self.play_button.toggled.connect(
            self.toggle_playback
        )

        controls.addWidget(
            self.play_button
        )

        self.previous_spike_button = (
            QtWidgets.QPushButton(
                "Previous spike"
            )
        )

        self.previous_spike_button.clicked.connect(
            self.previous_spike
        )

        controls.addWidget(
            self.previous_spike_button
        )

        self.next_spike_button = (
            QtWidgets.QPushButton(
                "Next spike"
            )
        )

        self.next_spike_button.clicked.connect(
            self.next_spike
        )

        controls.addWidget(
            self.next_spike_button
        )

        controls.addWidget(
            QtWidgets.QLabel(
                "Start:"
            )
        )

        self.start_time_box = (
            QtWidgets.QDoubleSpinBox()
        )

        self.start_time_box.setDecimals(
            3
        )

        self.start_time_box.setSingleStep(
            0.5
        )

        self.start_time_box.valueChanged.connect(
            self.start_time_changed
        )

        controls.addWidget(
            self.start_time_box
        )

        controls.addWidget(
            QtWidgets.QLabel(
                "X window:"
            )
        )

        self.window_box = (
            QtWidgets.QDoubleSpinBox()
        )

        self.window_box.setDecimals(
            2
        )

        self.window_box.setRange(
            0.05,
            1_000_000,
        )

        self.window_box.setSingleStep(
            1.0
        )

        self.window_box.setSuffix(
            " s"
        )

        self.window_box.valueChanged.connect(
            self.window_changed
        )

        controls.addWidget(
            self.window_box
        )

        controls.addWidget(
            QtWidgets.QLabel(
                "Playback step:"
            )
        )

        self.play_step_box = (
            QtWidgets.QDoubleSpinBox()
        )

        self.play_step_box.setRange(
            0.01,
            1000.0,
        )

        self.play_step_box.setDecimals(
            2
        )

        self.play_step_box.setValue(
            initial_play_step_seconds
        )

        self.play_step_box.setSuffix(
            " s"
        )

        controls.addWidget(
            self.play_step_box
        )

        controls.addWidget(
            QtWidgets.QLabel(
                "Signal:"
            )
        )

        self.signal_mode_box = (
            QtWidgets.QComboBox()
        )

        self.signal_mode_box.addItems([
            "Filtered",
            "Raw",
        ])

        self.signal_mode_box.setCurrentText(
            initial_signal_mode
        )

        self.signal_mode_box.currentTextChanged.connect(
            self.update_signal_mode
        )

        controls.addWidget(
            self.signal_mode_box
        )

        self.autoscale_y_checkbox = (
            QtWidgets.QCheckBox(
                "Auto Y"
            )
        )

        self.autoscale_y_checkbox.setChecked(
            True
        )

        self.autoscale_y_checkbox.toggled.connect(
            self.request_update
        )

        controls.addWidget(
            self.autoscale_y_checkbox
        )

        self.reset_button = (
            QtWidgets.QPushButton(
                "Reset"
            )
        )

        self.reset_button.clicked.connect(
            self.reset_view
        )

        controls.addWidget(
            self.reset_button
        )

        controls.addStretch()

        left_layout.addLayout(
            controls
        )

        self.time_slider = QtWidgets.QSlider(
            QtCore.Qt.Orientation.Horizontal
        )

        self.time_slider.setRange(
            0,
            100_000,
        )

        self.time_slider.valueChanged.connect(
            self.slider_changed
        )

        left_layout.addWidget(
            self.time_slider
        )

        self.status_label = QtWidgets.QLabel()

        left_layout.addWidget(
            self.status_label
        )

        self.graphics = (
            pg.GraphicsLayoutWidget()
        )

        left_layout.addWidget(
            self.graphics,
            stretch=1,
        )

        outer_layout.addWidget(
            left_widget,
            stretch=1,
        )

        # Right-side Y controls
        y_scroll = QtWidgets.QScrollArea()

        y_scroll.setWidgetResizable(
            True
        )

        y_scroll.setMinimumWidth(
            250
        )

        y_widget = QtWidgets.QWidget()

        self.y_control_layout = (
            QtWidgets.QVBoxLayout(
                y_widget
            )
        )

        self.y_control_layout.addWidget(
            QtWidgets.QLabel(
                "<b>Y-axis zoom</b><br>"
                "Above 1 zooms in.<br>"
                "Below 1 zooms out.<br><br>"
                "Mouse wheel also zooms.<br>"
                "Drag plots to pan."
            )
        )

        y_scroll.setWidget(
            y_widget
        )

        outer_layout.addWidget(
            y_scroll
        )

    # --------------------------------------------------------
    # Plot construction
    # --------------------------------------------------------

    def build_plots(self):
        for row, channel_name in enumerate(
            self.channel_names
        ):
            plot = self.graphics.addPlot(
                row=row,
                col=0,
            )

            plot.showGrid(
                x=True,
                y=True,
                alpha=0.2,
            )

            plot.setLabel(
                "left",
                channel_name,
            )

            if self.first_plot is None:
                self.first_plot = plot
            else:
                plot.setXLink(
                    self.first_plot
                )

            if row == len(
                self.channel_names
            ) - 1:
                plot.setLabel(
                    "bottom",
                    "Time",
                    units="s",
                )

            signal_line = pg.PlotDataItem(
                pen=pg.mkPen(
                    trace_color,
                    width=trace_width,
                )
            )

            # Peak-preserving rendering
            signal_line.setDownsampling(
                auto=True,
                method="peak",
            )

            signal_line.setClipToView(
                True
            )

            plot.addItem(
                signal_line
            )

            spike_items = {}

            for label in self.data_cache[
                channel_name
            ]["spikes"]:
                color = self.label_colors[
                    label
                ]

                scatter = pg.ScatterPlotItem(
                    size=spike_marker_size,
                    symbol="o",
                    pen=pg.mkPen(
                        color,
                        width=1.7,
                    ),
                    brush=pg.mkBrush(
                        0,
                        0,
                        0,
                        0,
                    ),
                )

                plot.addItem(scatter)

                spike_items[
                    label
                ] = scatter

            # Channel-specific Y zoom
            y_group = QtWidgets.QGroupBox(
                channel_name
            )

            y_layout = QtWidgets.QFormLayout(
                y_group
            )

            y_zoom = (
                QtWidgets.QDoubleSpinBox()
            )

            y_zoom.setRange(
                0.1,
                100.0,
            )

            y_zoom.setDecimals(
                2
            )

            y_zoom.setSingleStep(
                0.25
            )

            y_zoom.setValue(
                1.0
            )

            y_zoom.valueChanged.connect(
                self.request_update
            )

            y_layout.addRow(
                "Zoom:",
                y_zoom,
            )

            self.y_control_layout.addWidget(
                y_group
            )

            self.y_zoom_boxes[
                channel_name
            ] = y_zoom

            self.plot_records[
                channel_name
            ] = {
                "plot": plot,
                "line": signal_line,
                "spike_items": spike_items,
            }

        self.y_control_layout.addStretch()

    # --------------------------------------------------------
    # Recording ranges
    # --------------------------------------------------------

    def initialize_ranges(self):
        self.recording_start = min(
            self.data_cache[
                channel_name
            ]["time"][0]
            for channel_name
            in self.channel_names
        )

        self.recording_stop = max(
            self.data_cache[
                channel_name
            ]["time"][-1]
            for channel_name
            in self.channel_names
        )

        self.recording_duration = (
            self.recording_stop
            - self.recording_start
        )

        self.window_box.setMaximum(
            max(
                0.05,
                self.recording_duration,
            )
        )

        self.window_box.setValue(
            min(
                initial_window_seconds,
                self.recording_duration,
            )
        )

        self.start_time_box.setRange(
            self.recording_start,
            self.recording_stop,
        )

        self.start_time_box.setValue(
            self.recording_start
        )

    # --------------------------------------------------------
    # Keyboard shortcuts
    # --------------------------------------------------------

    def install_shortcuts(self):
        self.play_shortcut = (
            QtGui.QShortcut(
                QtGui.QKeySequence(
                    "Space"
                ),
                self,
            )
        )

        self.play_shortcut.activated.connect(
            lambda: self.play_button.setChecked(
                not self.play_button.isChecked()
            )
        )

        self.left_shortcut = (
            QtGui.QShortcut(
                QtGui.QKeySequence(
                    "Left"
                ),
                self,
            )
        )

        self.left_shortcut.activated.connect(
            lambda: self.shift_view(-0.25)
        )

        self.right_shortcut = (
            QtGui.QShortcut(
                QtGui.QKeySequence(
                    "Right"
                ),
                self,
            )
        )

        self.right_shortcut.activated.connect(
            lambda: self.shift_view(0.25)
        )

    # --------------------------------------------------------
    # Cached current signal
    # --------------------------------------------------------

    def current_mode(self):
        return (
            self.signal_mode_box
            .currentText()
        )

    def current_signal(
        self,
        channel_name,
    ):
        values = self.data_cache[
            channel_name
        ][self.current_mode()]

        if values is None:
            raise KeyError(
                f"{channel_name} does not contain "
                f"{self.current_mode()} data."
            )

        return values

    def update_signal_mode(self, *_):
        for channel_name in self.channel_names:
            channel_time = self.data_cache[
                channel_name
            ]["time"]

            channel_values = (
                self.current_signal(
                    channel_name
                )
            )

            self.plot_records[
                channel_name
            ]["line"].setData(
                channel_time,
                channel_values,
            )

        self.request_update()

    # --------------------------------------------------------
    # Debounced refresh
    # --------------------------------------------------------

    def request_update(self, *_):
        self.refresh_timer.start()

    # --------------------------------------------------------
    # Time conversion
    # --------------------------------------------------------

    def maximum_start_time(self):
        return max(
            self.recording_start,
            self.recording_stop
            - self.window_box.value(),
        )

    def slider_to_time(self, value):
        if (
            self.time_slider.maximum()
            <= 0
        ):
            return self.recording_start

        fraction = (
            value
            / self.time_slider.maximum()
        )

        return (
            self.recording_start
            + fraction
            * (
                self.maximum_start_time()
                - self.recording_start
            )
        )

    def time_to_slider(self, value):
        available = (
            self.maximum_start_time()
            - self.recording_start
        )

        if available <= 0:
            return 0

        fraction = (
            value
            - self.recording_start
        ) / available

        return int(
            np.clip(
                fraction,
                0,
                1,
            )
            * self.time_slider.maximum()
        )

    # --------------------------------------------------------
    # Time-control callbacks
    # --------------------------------------------------------

    def slider_changed(self, value):
        if self.updating_controls:
            return

        self.updating_controls = True

        new_time = self.slider_to_time(
            value
        )

        self.start_time_box.setValue(
            new_time
        )

        self.updating_controls = False
        self.request_update()

    def start_time_changed(self, value):
        if self.updating_controls:
            return

        self.updating_controls = True

        bounded_value = np.clip(
            value,
            self.recording_start,
            self.maximum_start_time(),
        )

        if bounded_value != value:
            self.start_time_box.setValue(
                bounded_value
            )

        self.time_slider.setValue(
            self.time_to_slider(
                bounded_value
            )
        )

        self.updating_controls = False
        self.request_update()

    def window_changed(self, *_):
        if self.updating_controls:
            return

        self.updating_controls = True

        current_start = min(
            self.start_time_box.value(),
            self.maximum_start_time(),
        )

        self.start_time_box.setValue(
            current_start
        )

        self.time_slider.setValue(
            self.time_to_slider(
                current_start
            )
        )

        self.updating_controls = False
        self.request_update()

    def shift_view(self, fraction):
        shift = (
            self.window_box.value()
            * fraction
        )

        new_start = np.clip(
            self.start_time_box.value()
            + shift,
            self.recording_start,
            self.maximum_start_time(),
        )

        self.start_time_box.setValue(
            new_start
        )

    # --------------------------------------------------------
    # Playback
    # --------------------------------------------------------

    def toggle_playback(self, playing):
        if playing:
            self.play_button.setText(
                "Pause"
            )

            self.play_timer.start()

        else:
            self.play_button.setText(
                "Play"
            )

            self.play_timer.stop()

    def advance_playback(self):
        new_start = (
            self.start_time_box.value()
            + self.play_step_box.value()
        )

        if (
            new_start
            > self.maximum_start_time()
        ):
            new_start = (
                self.recording_start
            )

        self.start_time_box.setValue(
            new_start
        )

    # --------------------------------------------------------
    # Previous and next spike
    # --------------------------------------------------------

    def center_on_time(self, spike_time):
        new_start = np.clip(
            spike_time
            - self.window_box.value() / 2,
            self.recording_start,
            self.maximum_start_time(),
        )

        self.start_time_box.setValue(
            new_start
        )

    def next_spike(self, *_):
        if not len(self.all_spike_times):
            return

        current_center = (
            self.start_time_box.value()
            + self.window_box.value() / 2
        )

        index = np.searchsorted(
            self.all_spike_times,
            current_center,
            side="right",
        )

        if index >= len(
            self.all_spike_times
        ):
            index = 0

        self.center_on_time(
            self.all_spike_times[index]
        )

    def previous_spike(self, *_):
        if not len(self.all_spike_times):
            return

        current_center = (
            self.start_time_box.value()
            + self.window_box.value() / 2
        )

        index = (
            np.searchsorted(
                self.all_spike_times,
                current_center,
                side="left",
            )
            - 1
        )

        if index < 0:
            index = (
                len(self.all_spike_times)
                - 1
            )

        self.center_on_time(
            self.all_spike_times[index]
        )

    # --------------------------------------------------------
    # Visible spike markers
    # --------------------------------------------------------

    @staticmethod
    def nearest_sample_indices(
        channel_time,
        spike_times,
    ):
        if (
            len(channel_time) < 2
            or len(spike_times) == 0
        ):
            return np.array(
                [],
                dtype=int,
            )

        indices = np.searchsorted(
            channel_time,
            spike_times,
        )

        indices = np.clip(
            indices,
            1,
            len(channel_time) - 1,
        )

        left_indices = indices - 1

        use_left = (
            np.abs(
                channel_time[left_indices]
                - spike_times
            )
            <
            np.abs(
                channel_time[indices]
                - spike_times
            )
        )

        indices[use_left] = (
            left_indices[use_left]
        )

        return indices

    def update_visible_spikes(
        self,
        channel_name,
        start_time,
        stop_time,
        channel_values,
    ):
        channel_cache = self.data_cache[
            channel_name
        ]

        channel_time = channel_cache[
            "time"
        ]

        spike_items = self.plot_records[
            channel_name
        ]["spike_items"]

        visible_count = 0

        for label, scatter in (
            spike_items.items()
        ):
            all_times = channel_cache[
                "spikes"
            ][label]

            left = np.searchsorted(
                all_times,
                start_time,
                side="left",
            )

            right = np.searchsorted(
                all_times,
                stop_time,
                side="right",
            )

            visible_times = all_times[
                left:right
            ]

            if (
                len(visible_times)
                > maximum_visible_markers_per_label
            ):
                stride = int(
                    np.ceil(
                        len(visible_times)
                        / maximum_visible_markers_per_label
                    )
                )

                visible_times = (
                    visible_times[::stride]
                )

            if not len(visible_times):
                scatter.setData(
                    x=[],
                    y=[],
                )
                continue

            sample_indices = (
                self.nearest_sample_indices(
                    channel_time,
                    visible_times,
                )
            )

            marker_values = channel_values[
                sample_indices
            ]

            valid = (
                np.isfinite(visible_times)
                & np.isfinite(marker_values)
            )

            scatter.setData(
                x=visible_times[valid],
                y=marker_values[valid],
            )

            visible_count += int(
                np.sum(valid)
            )

        return visible_count

    # --------------------------------------------------------
    # Main view update
    # --------------------------------------------------------

    def update_view(self):
        start_time = float(
            np.clip(
                self.start_time_box.value(),
                self.recording_start,
                self.maximum_start_time(),
            )
        )

        stop_time = min(
            start_time
            + self.window_box.value(),
            self.recording_stop,
        )

        # Linked X axes only require updating the first plot
        self.first_plot.setXRange(
            start_time,
            stop_time,
            padding=0,
        )

        total_visible_spikes = 0

        for channel_name in self.channel_names:
            channel_cache = self.data_cache[
                channel_name
            ]

            channel_time = channel_cache[
                "time"
            ]

            channel_values = (
                self.current_signal(
                    channel_name
                )
            )

            left_index = np.searchsorted(
                channel_time,
                start_time,
                side="left",
            )

            right_index = np.searchsorted(
                channel_time,
                stop_time,
                side="right",
            )

            visible_values = channel_values[
                left_index:right_index
            ]

            total_visible_spikes += (
                self.update_visible_spikes(
                    channel_name,
                    start_time,
                    stop_time,
                    channel_values,
                )
            )

            if (
                self.autoscale_y_checkbox
                .isChecked()
            ):
                finite_values = (
                    visible_values[
                        np.isfinite(
                            visible_values
                        )
                    ]
                )

                if len(finite_values):
                    low, high = np.percentile(
                        finite_values,
                        [
                            y_percentile_low,
                            y_percentile_high,
                        ],
                    )

                    center = (
                        low + high
                    ) / 2

                    half_range = (
                        high - low
                    ) / 2

                    if half_range <= 0:
                        half_range = max(
                            abs(center) * 0.05,
                            1e-12,
                        )

                    zoom = (
                        self.y_zoom_boxes[
                            channel_name
                        ].value()
                    )

                    shown_half_range = (
                        half_range / zoom
                    )

                    self.plot_records[
                        channel_name
                    ]["plot"].setYRange(
                        center
                        - shown_half_range,
                        center
                        + shown_half_range,
                        padding=0,
                    )

        self.status_label.setText(
            f"Showing {start_time:.3f}–"
            f"{stop_time:.3f} s | "
            f"{total_visible_spikes:,} visible markers | "
            f"{self.current_mode()} signal"
        )

    # --------------------------------------------------------
    # Reset
    # --------------------------------------------------------

    def reset_view(self, *_):
        self.play_timer.stop()
        self.play_button.setChecked(
            False
        )

        self.start_time_box.setValue(
            self.recording_start
        )

        self.window_box.setValue(
            min(
                initial_window_seconds,
                self.recording_duration,
            )
        )

        self.play_step_box.setValue(
            initial_play_step_seconds
        )

        self.autoscale_y_checkbox.setChecked(
            True
        )

        for zoom_box in (
            self.y_zoom_boxes.values()
        ):
            zoom_box.setValue(
                1.0
            )

        self.update_view()


# ============================================================
# OPEN WINDOW
# ============================================================

app = QtWidgets.QApplication.instance()

if app is None:
    app = QtWidgets.QApplication(
        sys.argv
    )

# Close the previous viewer if this cell is rerun
if (
    "sorted_spike_viewer_window"
    in globals()
):
    try:
        sorted_spike_viewer_window.close()
    except Exception:
        pass

sorted_spike_viewer_window = (
    SortedSpikeViewer(
        processed_channels=processed_channels,
        spikes_by_channel=(
            sorted_spikes_by_channel
        ),
        channel_names=(
            channels_to_display
        ),
        label_colors=label_colors,
    )
)

sorted_spike_viewer_window.show()
sorted_spike_viewer_window.raise_()
sorted_spike_viewer_window.activateWindow()